# You Can Only Run This Code On Colab

# Installs
## run this twice the first time wait for the restart window to pop-up and second time just run it normal

In [ ]:
# import transformers

# transformers.__version__

In [ ]:
# !pip install rembg
# !pip install onnxruntime
# !pip install git+https://github.com/facebookresearch/segment-anything.git

# !pip install Flask pyngrok

# !pip install celery


# !pip install transformers==4.53.0
# !pip install thefuzz[speedup]
# !pip install webcolors

# # !pip install --upgrade torch torchvision

# Getting the Attributes
## You have to upload an image named image.png to or change the path name in Run Function but keep the extension .png


## IMPORTS

In [ ]:
from transformers import CLIPProcessor, CLIPModel, AutoProcessor, AutoModelForCausalLM
from PIL import Image
import torch
import warnings
warnings.filterwarnings("ignore")
from rembg import remove
import requests
import os
import numpy as np
import re
import glob
from typing import Tuple

from sklearn.cluster import KMeans
import colorsys
import json
import gc

from thefuzz import fuzz, process

import webcolors

## LISTS

In [2]:
HIGH_LEVEL_CATEGORY_DESCRIPTIONS = {
    "Top": "A garment that covers the upper body, such as a shirt, sweater, or blouse.",
    "Outerwear": "A garment worn over clothes for warmth or protection, such as a jacket, coat, or vest.",
    "Bottom": "A garment that covers the lower body and legs, such as pants, shorts, or trousers.",
    "Dress/Jumpsuit": "A single-piece garment that combines a top and bottom, such as a dress or jumpsuit.",
    "Skirt": "A garment that covers the body from the waist down, often in an A-line, pleated, or pencil style.",
    "Shoes": "Footwear worn on the feet, such as boots, sneakers, sandals, or heels.",
    "Underwear": "A garment worn closest to the body, such as a bra, panty, or boxers.",
}

CATEGORY_DESCRIPTIONS = {
    "Blouse": "A stylish and often elegant top, typically worn by women, characterized by its loose fit, soft fabrics like silk, chiffon, or cotton, and decorative elements such as ruffles, lace, embroidery, or bows. It can be worn for both casual and formal occasions, often paired with skirts, trousers, or jeans. Blouses may feature various sleeve lengths, necklines, and closures, including buttons, ties, or pullovers.",
    "Button-down shirt": "A versatile collared shirt with a full-length vertical opening at the front, fastened with a series of buttons. It typically features a pointed collar, long or short sleeves with cuffs, and a structured fit. Made from a variety of fabrics like cotton, linen, or flannel, it's suitable for formal, business-casual, or casual wear, often tucked in or worn untucked.",
    "Button-up shirt": "button-up shirt",
    "Crop top": "A fashionable top designed to expose the midriff, waist, or navel. Its hemline sits notably above the natural waist. Crop tops come in various styles, materials, and sleeve lengths, from fitted to loose, and can be worn casually or dressed up, often paired with high-waisted bottoms to balance the silhouette.",
    "Henley shirt": "A distinctive collarless pullover shirt with a round neckline and a short placket, typically 3 to 5 inches long, fastened by two to five buttons. It offers a more refined look than a standard T-shirt but maintains a casual comfort, often made from cotton or a cotton blend. Henleys can be long-sleeved or short-sleeved and are popular for everyday wear.",
    "Jersey": "A soft, flexible, and comfortable knitted garment, typically a pullover shirt, made from wool, cotton, or synthetic blends. 'Jersey' can refer to a casual everyday sweater or, more commonly, to a distinctive sports uniform top worn by athletes to identify team members, often featuring numbers, team names, and logos. It's designed for comfort and mobility.",
    "Knit sweater": "A warm and cozy top crafted from interlocking loops of yarn through knitting or crocheting, typically made of wool, cashmere, cotton, or synthetic fibers. Sweaters are worn for warmth and comfort, coming in various styles like pullovers (crewneck, V-neck, turtleneck) or cardigans (button-front). They can range from chunky, casual designs to fine-gauge, more formal options.",
    "Polo shirt": "A classic smart-casual shirt featuring a collar, a placket with two or three buttons, and typically short sleeves with ribbed cuffs. Originally designed for polo players, it's made from a durable, breathable knitted fabric, often pique cotton. The polo shirt bridges the gap between a T-shirt and a button-down, suitable for various informal and semi-formal settings.",
    "Pullover hoodie": "A casual, hooded sweatshirt without a zipper, designed to be pulled over the head. It typically features a large kangaroo pocket on the front and drawstrings to adjust the hood opening. Made from fleece or cotton blends, it's a popular choice for comfort, warmth, and relaxed style, widely worn for athletic activities or everyday casual wear.",
    "Sleeveless top": "A broad category of upper-body garments that lack sleeves, distinguishing them from traditional T-shirts or blouses by having exposed shoulders and arms. This can include a variety of styles beyond just tank tops, such as sleeveless blouses, vests, or more structured tops with higher necklines and different armhole cuts. They are popular in warm weather or as layering pieces.",
    "Tank top": "A specific type of sleeveless shirt characterized by large armholes, a low neckline, and narrow shoulder straps. Made from lightweight, breathable fabrics like cotton, it's commonly worn as casual wear, athletic wear, or as an undershirt, particularly popular in warm climates or for physical activity.",
    "T-shirt": "A fundamental casual garment, consisting of a simple, collarless short-sleeved shirt with either a round (crew neck) or V-shaped neckline. Typically made from soft, breathable cotton or cotton blends, it's a ubiquitous item of casual wear, versatile enough for everyday comfort, layering, or as a base for graphic designs.",
    "Tunic": "A loose-fitting, often lightweight garment that is longer than a typical top, extending to or past the hips, and sometimes to the knees. It can be sleeved or sleeveless, featuring various necklines and often worn over leggings, pants, or skirts. Tunics are popular for their comfortable fit and flattering drape, adaptable for both casual and semi-formal wear.",
    "Turtleneck": "A style of sweater or shirt characterized by a high, close-fitting, cylindrical collar that folds over itself to cover the neck. Turtlenecks are typically made from knitted materials like wool, cotton, or synthetic blends, providing warmth and a distinctively chic or intellectual look. They can be worn alone or as a layering piece under jackets and blazers.",

    "Blazer": "A sophisticated jacket resembling a suit jacket but cut more casually for standalone wear rather than as part of a matching suit. Often made from wool, flannel, or cotton, blazers feature a notched lapel, front buttons, and pockets, sometimes with contrasting buttons or piping. They are highly versatile, perfect for smart-casual, business-casual, or even dressed-up casual outfits.",
    "Cardigan sweater": "A versatile knitted sweater that opens fully down the front, secured with buttons, a zipper, or left open. Cardigans come in various gauges, lengths, and materials, from lightweight cotton to chunky wool. They serve as an excellent layering piece, offering warmth and style for a range of outfits, from casual to semi-formal.",
    "Denim jacket": "A timeless piece of casual outerwear made entirely from sturdy denim fabric, often featuring a pointed collar, button-front closure, two chest pockets with flaps, and vertical seams. Known for its durability and ability to age beautifully, it's a versatile layering item that complements a wide array of casual outfits.",
    "Leather jacket": "A classic and iconic piece of outerwear crafted from treated animal hide, offering durability, style, and warmth. Common styles include the rugged 'biker' jacket (often with zippers and asymmetrical closures), the sleek 'bomber' jacket (with ribbed cuffs and hem), and tailored 'blazer' styles. Leather jackets are highly fashionable and associated with a cool, edgy aesthetic.",
    "Parka": "A heavy, warm, and often waterproof jacket designed for cold weather conditions, typically extending to the hip or thigh. Parkas usually feature a hood, often lined with fur or faux fur, and are insulated with down or synthetic fill. They are known for their windproof and weather-resistant properties, making them ideal for extreme cold.",
    "Peacoat": "A short, double-breasted overcoat traditionally made of heavy, dense wool, known for its warmth and durability. It features a broad lapel, vertical slit pockets, large buttons, and is often navy blue. Originally worn by sailors, the peacoat is a timeless, classic piece of outerwear suitable for cold weather, offering a smart and rugged look.",
    "Puffer jacket": "A quilted, insulated jacket characterized by its distinctive 'puffy' sections, which are filled with down feathers or synthetic fibers for superior warmth and lightness. These jackets often have a shiny or matte outer shell, a zipper front, and can include a hood. Puffer jackets are highly effective for cold weather and are popular for both urban and outdoor use.",
    "Raincoat": "A functional outer garment specifically designed to be waterproof or water-resistant, protecting the wearer from rain and wet weather. Raincoats are made from materials like treated cotton, nylon, or PVC, and often feature sealed seams, hoods, and protective closures to keep moisture out. They range from lightweight, packable styles to more robust, fashionable designs.",
    "Leather coat": "A long, loose coat, often with a back vent and sometimes a shoulder cape, originally designed to protect from dust. Now worn as dramatic, stylish outerwear.",    "Trench coat": "A sophisticated and classic long coat, typically double-breasted with a belt, shoulder epaulets, and often a storm flap. Traditionally made from waterproof gabardine cotton, it's known for its elegant drape and timeless design. Trench coats are versatile outerwear, suitable for various weather conditions and can be dressed up or down.",
    "Vest": "A sleeveless upper-body garment that can be worn as part of a suit (over a dress shirt and under a suit jacket) or as a standalone piece of outerwear. Vests come in various styles, including tailored suit vests, padded or quilted vests for warmth (puffer vests), and functional utility vests with multiple pockets. They add a layer of warmth or style without bulky sleeves.",
    "Windbreaker jacket": "A lightweight, thin jacket primarily designed to resist wind chill and light rain, making it ideal for transitional weather or active pursuits. Typically made from synthetic materials like nylon or polyester, windbreakers are often packable and feature elasticated cuffs and hems, and sometimes a hood. They offer minimal insulation but excellent wind protection.",
    "Zippered hoodie": "A casual hooded sweatshirt that features a full-length zipper down the front, allowing it to be easily worn open or closed. It typically includes a hood with drawstrings and often front pockets. Made from fleece or cotton blends, it's a comfortable and versatile layering piece for athletic activities or everyday wear, offering adjustable warmth.",
    "Tracksuit jacket": "A sporty jacket typically made from smooth synthetic fabrics like polyester or nylon, featuring a full‑length front zipper, elasticated cuffs and hem, and side pockets. Often accented with contrasting stripes along the sleeves and shoulders, it has a relaxed fit ideal for athletic warm‑ups or casual streetwear.",
    "Leather bomber jacket": "A waist-length leather jacket with a zip front, ribbed cuffs and hem, and a relaxed fit. Originally designed for pilots, it often features a ribbed or shearling collar and is known for its classic, rugged style.",
    "Coat": "A general term for a long-sleeved outerwear garment that extends below the waist, designed to provide warmth and protection from the elements. Coats can feature various closures such as buttons or zippers, and come in a range of materials including wool, cotton, or synthetics. Styles may include single- or double-breasted fronts, different collar types, and a variety of patterns or color-block designs. Suitable for both casual and formal occasions.",
    "Jacket": "A versatile outerwear garment that typically extends to or just below the waist, designed to provide light to moderate warmth and protection. Jackets come in various styles, such as denim, leather, or windbreaker, and feature closures like zippers, buttons, or snaps. They are suitable for casual wear and can be layered with other clothing for added warmth.",

    "Bootcut jeans": "Jeans designed to be fitted through the thigh and then subtly flare out from the knee down to the ankle, allowing them to be comfortably worn over boots. This cut creates a balanced silhouette and is a classic style, often made from traditional denim fabric with various washes.",
    "Capri pants": "Trousers that are shorter than full-length pants but longer than shorts, typically ending at the mid-calf. Capri pants are a casual and comfortable option, often made from cotton, linen, or stretch fabrics, popular in warmer weather or for relaxed activities.",
    "Cargo pants": "Practical and loose-fitting trousers characterized by one or more large, distinctive patch pockets located on the sides of the legs, often with bellows for expanded capacity and secured with flaps. Originally designed for military utility, they are now popular casual wear, made from durable fabrics like cotton or ripstop.",
    "Chino pants": "Versatile casual trousers made from chino cloth, a lightweight cotton twill fabric. Chinos typically feature a flat front, slanted side pockets, and jetted back pockets, offering a neat yet relaxed appearance. They bridge the gap between jeans and dress pants, suitable for smart-casual and everyday wear.",
    "Sweatpants": "Casual, loose-fitting pants made from soft, comfortable fabrics like cotton or fleece. Sweatpants typically feature an elastic or drawstring waistband and elastic cuffs at the ankles. They are designed for comfort and ease of movement, making them popular for athletic activities, lounging, or casual wear.",
    "Dress pants": "Tailored trousers intended for formal or semi-formal wear, often made from fine fabrics like wool, gabardine, or blends, and typically featuring pleats or a flat front, belt loops, and a straight or slightly tapered leg. They are frequently worn as part of a suit or paired with a dress shirt and blazer for professional or special occasions.",
    "Joggers": "Casual and comfortable pants characterized by a relaxed fit through the hips and thighs, with a distinct taper towards the ankle, ending in an elasticated or ribbed cuff. They typically feature an elasticated or drawstring waistband, often made from soft fabrics like fleece or jersey, popular for athletic activities or relaxed everyday wear.",
    "Leggings": "Tight-fitting, form-hugging stretch pants, typically worn by women and girls, extending from the waist down to the ankle. Made from stretchy fabrics like spandex, cotton blends, or synthetic materials, they are popular for athletic activities, casual wear, or as a layering piece under skirts and dresses.",
    "Shorts": "A garment covering the pelvic area and the upper part of the legs, but not the entire leg, with varying lengths from very short to knee-length. Shorts are primarily worn in warm weather or for athletic activities, made from a wide range of fabrics like denim, cotton, linen, or technical materials.",
    "Skinny jeans": "Jeans characterized by a very close, tight fit from the waist through the hips, thighs, and all the way down to the ankle. They are designed to hug the leg's natural contours, often made with stretch denim for comfort and flexibility. Skinny jeans are a popular modern style for casual wear.",
    "Straight-leg jeans": "Classic jeans that maintain a consistent, straight width from the thigh down to the ankle, creating a uniform silhouette that doesn't flare or taper significantly. This timeless cut offers a relaxed yet neat appearance and is a versatile staple in casual wardrobes.",
     "Trousers":"Trousers",
    "Denim shorts": (
    "Shorts made from sturdy denim twill fabric, "
    "typically mid‑thigh length, often featuring a button or zipper "
    "waistband, front/back pockets, and sometimes distressed or frayed hems. "
    "They combine the durability and classic blue‑jeans aesthetic with the "
    "casual comfort of shorts."

),
    "A-line skirt": "A flattering skirt that is fitted at the waist and hips and gradually widens towards the hem, creating a shape resembling the capital letter 'A'. This silhouette is universally flattering and comes in various lengths and fabrics, suitable for both casual and more formal settings.",
    "Denim skirt": "A casual skirt made from durable denim fabric, similar to that used for jeans. Denim skirts come in various lengths (mini, midi, maxi) and styles (A-line, pencil, button-front), often featuring characteristic denim details like topstitching, rivets, and pockets. They offer a versatile, relaxed, and often rugged aesthetic.",
    "Maxi skirt": "A long skirt with a hemline that typically falls to the ankle or the floor. Maxi skirts are known for their comfortable, flowing silhouette and are often made from lightweight, breathable fabrics. They are popular for bohemian, casual, or elegant looks, especially in warmer weather.",
    "Mini skirt": "A very short skirt with a hemline that falls significantly above the knee, typically at mid-thigh level or higher. Mini skirts are a bold fashion statement, designed to showcase the legs, and come in a wide range of fabrics and styles, from casual denim to more formal tailored versions.",
    "Pencil skirt": "A slim-fitting skirt with a straight, narrow cut that closely follows the lines of the body, extending usually to or just below the knee. Named for its sleek, elongated shape, it's a classic and sophisticated style often associated with professional or formal wear, providing a chic and structured silhouette.",
    "Pleated skirt": "A skirt distinguished by its fabric being folded and pressed into permanent pleats, creating a structured and often voluminous look. Pleated skirts can feature knife pleats, box pleats, or accordion pleats, and vary in length from mini to maxi, offering a dynamic and elegant movement.",
    "Wrap skirt": "A skirt designed to wrap around the wearer's waist, with one panel overlapping the other, and secured by ties, buttons, or a buckle. This construction allows for an adjustable fit and creates a natural slit or opening when worn. Wrap skirts are versatile, comfortable, and come in various lengths and fabrics, suitable for casual or semi-formal occasions.",

    "Cocktail dress": "A semi-formal dress typically worn for cocktail parties and other semi-formal occasions, usually characterized by a length that falls above the ankle, often knee-length or slightly shorter. Cocktail dresses are designed to be elegant and stylish, featuring various necklines, sleeve styles, and embellishments, and are more ornate than day dresses but less formal than evening gowns.",
    "Evening gown": "A long, formal dress typically worn by women to very formal events such as black-tie galas, balls, and formal dinners. Evening gowns are characterized by their floor-length hemlines, luxurious fabrics (silk, satin, velvet, lace), and elegant designs, often featuring intricate embellishments, dramatic silhouettes, or sophisticated draping.",
    "Jumpsuit": "A one-piece garment that combines a top (like a blouse or shirt) and trousers into a single, seamless outfit. Jumpsuits are versatile, offering a complete look with minimal effort, and come in a wide range of styles, from casual utilitarian designs to elegant, tailored versions suitable for formal events.",
    "Maxi dress": "A long, informal dress with a hemline that extends to the ankle or floor. Maxi dresses are known for their comfortable, flowing silhouette, often made from lightweight and breathable fabrics. They are a popular choice for casual wear in warm weather, beach outings, or relaxed daytime events, offering effortless style.",
    "Midi dress": "A dress with a hemline that falls somewhere between the knee and the ankle, typically mid-calf. Midi dresses offer a sophisticated and versatile length, suitable for a wide range of occasions from casual to semi-formal, depending on the fabric and style. They are a popular modern choice, providing elegance without being overly formal.",
    "Mini dress": "A very short dress with a hemline that is notably above the knees, typically at mid-thigh level or higher. Mini dresses are designed to be playful and bold, showcasing the legs. They come in various styles, fabrics, and fits, popular for casual outings, parties, and fashion statements.",
    "Overalls": "A one-piece garment consisting of trousers with a bib section attached, held up by straps over the shoulders. Traditionally made of durable denim and associated with workwear, overalls are also popular casual fashion items, offering a relaxed and utilitarian style. They often feature multiple pockets on the bib and trousers.",
    "Romper": "A one-piece garment that combines a top with shorts, creating a cohesive and often playful outfit. Rompers are typically lightweight and designed for comfort, popular in warm weather for casual outings, beachwear, or relaxed everyday wear. They come in various necklines, sleeve styles, and fabrics.",
    "Shirt dress": "A style of dress that borrows design elements from a man's button-down shirt, such as a collar, a full-length button front, and shirt-like sleeves with cuffs. Shirt dresses range from casual, loose-fitting styles to more tailored versions, sometimes with a belt to cinch the waist, offering a blend of comfort and structured elegance.",
    "Strap dress": "A dress characterized by its reliance on narrow straps over the shoulders to support the garment, leaving the upper chest, back, and shoulders exposed. Often made from lightweight fabrics, strap dresses are popular for warm weather, casual outings, or as a base for layering with cardigans or jackets. They can range from simple sundresses to more elegant slip dresses.",
    "Sundress": "A light, loose-fitting, and casual dress specifically designed for warm weather. Sundresses are typically sleeveless, often feature thin straps, and are made from breathable fabrics like cotton or linen. They are comfortable and airy, perfect for daytime activities, outdoor events, and relaxed summer styles.",
    "Sweater dress": "A dress made from knitted material, similar to a sweater, designed to be worn as a complete outfit. Sweater dresses vary in thickness, length (mini to maxi), and knit patterns, offering warmth and comfort while maintaining a stylish silhouette. They are a popular choice for fall and winter wear.",
    "T-shirt dress": "A casual dress styled like an elongated T-shirt, typically loose-fitting and comfortable, with a simple crew neck or V-neck and short sleeves. Made from soft cotton or jersey fabric, T-shirt dresses are ideal for relaxed everyday wear, offering effortless style and versatility.",
    "Wrap dress": "A dress with a front closure formed by wrapping one side of the dress across the other, tying or fastening the attached ties around the back at the waist. This design creates a flattering V-neckline and an adjustable, comfortable fit. Wrap dresses are highly versatile, suitable for various body types and occasions, from casual to semi-formal.",
    "Denim dress": "A casual dress made from durable denim fabric, similar to that used for jeans. Denim dresses come in various lengths (mini, midi, maxi) and styles (A-line, shirt dress, button-front), often featuring characteristic denim details like topstitching, rivets, and pockets. They offer a versatile, relaxed, and often rugged aesthetic.",
    "Dress":"dress",

    "Ankle boots": "Boots that extend up to or just above the ankle, covering the foot and ankle but no further up the leg. Ankle boots are a highly versatile footwear option, coming in a vast array of styles, heel heights, and materials, suitable for various seasons and outfits, from casual to dressy.",
    "Ballet flats": "Flat, lightweight shoes inspired by a ballerina's soft slipper, characterized by a thin sole and typically no heel or a very low heel. They often feature a rounded toe and a simple, slip-on design. Ballet flats are a comfortable and elegant choice for casual or smart-casual wear.",
    "Espadrilles": "Casual, lightweight shoes characterized by their distinctive rope (esparto fiber) sole, often with a canvas or cotton fabric upper. Espadrilles can be flat, wedge, or platform, and sometimes feature ankle ties. They are a popular choice for warm weather, offering a relaxed and summery aesthetic.",
    "Heel pumps": "A classic type of high-heeled shoe characterized by a low-cut front (vamp) that exposes the top of the foot, and a closed back. Pumps do not have straps or fastenings and rely on the fit for secure wear. They come in various heel heights and are a staple for formal, business, or evening wear, adding elegance and height.",
    "Knee-high boots": "Boots that extend up to the knee, or sometimes slightly below or over it. These boots offer significant coverage and can be made from various materials like leather, suede, or synthetic fabrics. Knee-high boots are a fashionable choice for cold weather, often paired with skirts, dresses, or skinny jeans, providing warmth and a stylish statement.",
    "Loafers": "A comfortable, slip-on style of shoe with no lacing or other fastenings. Loafers are typically low-cut and can have a flat sole or a very low heel. They are a popular choice for smart-casual wear, offering ease of wear and a polished yet relaxed appearance, often featuring decorative elements like tassels or metal bits.",
    "Oxford dress shoes": "A traditional and formal style of leather shoe characterized by its 'closed lacing' system, where the shoelace eyelet tabs are stitched underneath the vamp. Oxfords typically have a low heel and are known for their sleek, elegant appearance, making them a staple for business, formal, and dressy casual occasions.",
    "Rain boots": "Practical, waterproof boots designed specifically to protect the feet from rain, mud, and wet conditions. They are typically made from rubber or PVC and often extend to mid-calf or knee-high. Rain boots come in various colors and patterns, providing essential protection while also being a fun fashion accessory.",
    "Sandals": "Open-type footwear consisting of a sole held to the wearer's foot by straps or bands over the instep, and sometimes around the ankle or toes. Sandals are designed for warm weather, allowing the foot to remain largely exposed. They come in a vast range of styles, from casual flip-flops to elegant heeled designs.",
    "Sneakers": "Athletic-inspired shoes primarily designed for sports, physical exercise, or general casual wear. Sneakers feature a flexible sole made of rubber or synthetic material and an upper made of leather, canvas, or synthetic fabric. They are known for their comfort and support, and have become a ubiquitous item in everyday casual fashion.",
    "Slippers": "Soft, comfortable slip-on shoes primarily designed for indoor wear, providing warmth and comfort around the house. Slippers are typically made from soft fabrics like fleece, wool, or memory foam, and often have a soft, non-slip sole. They prioritize coziness and ease of wear.",
    "Wedges": "Shoes or boots that feature a sole in the form of a wedge, where one piece of material serves as both the sole and the heel, creating a continuous rise from the front to the back of the foot. Wedges offer more stability and comfort than traditional high heels while still providing height, popular in sandals, espadrilles, and boots.",
    "Suede dress shoes": "A refined style of men's formal footwear crafted from soft suede leather, characterized by a closed-lacing system where the eyelet tabs are stitched under the vamp. These shoes feature a smooth, rounded toe, a low heel, and a clean, minimalist silhouette. The suede material adds a luxurious texture and a slightly more relaxed feel compared to polished leather Oxfords, making them perfect for formal occasions, business attire, or dressing up smart-casual outfits.",
    "Dress shoe":"A classic men's black leather dress shoe featuring a smooth, polished upper and a refined round toe. It includes matching black laces neatly tied at the front and a low-profile heel for a formal, timeless appearance. The sole is crafted from durable material in a coordinating black tone, offering both comfort and sophistication ,perfect for business, formal events, or elegant evening wear.",


    "Baseball cap": "A soft cap with a rounded crown and a stiff peak projecting forward, designed to shade the eyes from the sun. Originally associated with baseball players, it's now a widely popular casual headwear item, often featuring logos or embroidered designs. Baseball caps are adjustable and worn for athletic, casual, or fashion purposes.",
    "Beanie hat": "A close-fitting, brimless cap, typically made from knitted material like wool or acrylic. Beanies are designed to provide warmth and cover the head and ears, often worn in cold weather or as a casual fashion accessory. They can be worn tight to the head or with a slight slouch.",
    "Beret": "A soft, round, flat-crowned cap, typically of wool felt or hand-knitted wool. Berets are brimless and traditionally associated with artists, military personnel, and certain European cultures. They are worn for both practical warmth and as a chic fashion accessory, often tilted to one side.",
    "Bucket hat": "A soft cotton hat with a wide, downward-sloping brim that resembles a bucket, providing all-around sun protection. Originally worn by fishermen and farmers, it became a popular fashion accessory in the 1980s and has seen various revivals. Bucket hats are casual, foldable, and versatile.",
    "Fedora": "A classic felt hat characterized by an indented crown (typically pinched at the front and creased lengthwise down the crown) and a soft, typically wider brim. Fedoras are often made of felt, wool, or straw and are associated with a sophisticated, vintage, or classic style, worn in both casual and formal contexts.",
    "Fez hat": "A traditional, flat-topped, conical red felt hat with a black tassel hanging from the crown, worn by men in some Muslim countries, particularly in North Africa and the Middle East. It is a symbol of cultural heritage and can be worn as everyday headwear or for ceremonial purposes.",
    "Headband": "A band made of fabric, plastic, or metal worn around the head. Headbands serve various purposes, including holding hair back from the face, absorbing sweat during sports, or as a decorative fashion accessory. They come in numerous styles, widths, and materials.",
    "Headscarf": "A square or triangular piece of cloth worn wrapped around the head. Headscarves can serve various purposes, including religious observance (e.g., hijab), protection from the elements, or as a fashion accessory. They come in diverse fabrics, patterns, and tying styles.",
    "Hijab head covering": "A veil or head covering worn by some Muslim women in the presence of any male outside of their immediate family, typically covering the head and chest. It is a symbol of modesty and privacy, deeply rooted in religious and cultural traditions, and comes in various styles, fabrics, and colors.",
    "Sombrero": "A broad-brimmed hat, typically made of straw or felt, with a high, pointed or conical crown. Originating from Mexico, it's designed to provide ample shade from the sun. Sombreros are iconic traditional headwear, often decorated, and worn for cultural events or as part of traditional dress.",
    "Sun hat": "A hat specifically designed with a wide brim to provide extensive shade for the face, neck, and ears, protecting against harsh sunlight. Sun hats are typically made from lightweight, breathable materials like straw, cotton, or canvas, and are popular for outdoor activities, beachwear, and gardening.",
    "Turban": "A type of headwear consisting of a long length of fabric, typically cotton or silk, wound around the head or a cap. Turbans are worn by men and women in various cultures around the world, particularly in parts of Asia, Africa, and the Middle East, often for religious reasons, cultural identity, or as a fashion statement.",

    "Ankle socks": "Short socks that extend only to or just above the ankle, making them minimally visible when worn with most shoes. They are popular for casual wear and athletic activities, providing comfort and preventing shoe rub while maintaining a low profile.",
    "Backpack": "A highly functional bag designed to be carried on one's back, supported by two shoulder straps. Backpacks come in various sizes and designs, from small daypacks to large hiking packs, and are used for carrying books, travel essentials, gear, or everyday items. They are known for their ergonomic weight distribution.",
    "Bandana": "A square or triangular piece of cloth, often patterned with designs like paisley, typically made of cotton. Bandanas are versatile accessories worn around the head (as a headband or head covering), around the neck, or tied to bags for decoration. They are a classic, casual, and sometimes rugged accessory.",
    "Belt": "A flexible band or strap, typically made of leather, fabric, or synthetic material, worn around the waist. Belts serve both functional purposes (holding up trousers or skirts) and decorative ones (cinching a dress or adding a stylish accent). They come in various widths, colors, and buckle designs.",
    "Bow tie": "A type of necktie that consists of a ribbon of fabric tied around the collar of a shirt in a symmetrical manner, forming two loops or 'bows.' Bow ties are a classic and formal accessory, often worn with tuxedos, suits, or dress shirts, adding a distinctive and sophisticated touch.",
    "Bracelet": "An ornamental band, hoop, or chain worn on the wrist or arm. Bracelets come in an extensive variety of materials (metal, beads, leather, fabric) and designs, from simple bands to elaborate pieces, serving as a personal adornment or symbol.",
    "Brooch": "An ornamental clasp or pin fastened to clothing, typically to a dress, coat, or scarf, with a hinged pin and catch. Brooches often feature intricate designs, gemstones, or enamel work, serving as a decorative accent or a statement piece of jewelry.",
    "Clutch bag": "A small, strapless handbag designed to be carried or 'clutched' in the hand, or sometimes tucked under the arm. Clutches are typically elegant and compact, intended for carrying only essential items like a phone, keys, and cards, commonly used for evening wear or formal events.",
    "Crew socks": "Socks of a medium length, typically extending a few inches above the ankle but below the calf. Crew socks are a versatile and common length for everyday wear, providing comfort and moderate coverage, suitable for various types of shoes.",
    "Crossbody bag": "A bag with a long strap designed to be worn across the body, with the bag resting on the opposite hip. This style offers security and hands-free convenience, making it popular for travel, shopping, or everyday use. Crossbody bags come in various sizes and materials.",
    "Dress socks": "Socks specifically intended to be worn with formal or business attire, such as suits and dress shoes. Dress socks are typically thin, often made from fine materials like cotton, silk, or wool blends, and come in darker, solid colors or subtle patterns to complement formal wear.",
    "Earbuds": "Very small headphones that are placed directly into the outer ear or ear canal, designed for portable audio listening. Earbuds are compact and discreet, making them highly convenient for on-the-go use with smartphones, MP3 players, and other devices.",
    "Earrings": "Jewelry worn on the earlobe or another external part of the ear, typically through a piercing. Earrings come in an immense variety of designs, from simple studs to elaborate dangling pieces, made from various materials, serving as a popular form of personal adornment.",
    "Eyeglasses": "A pair of lenses set in a frame that rests on the nose and ears, used to correct or assist defective eyesight, or sometimes worn as a fashion accessory without prescription lenses. Eyeglasses are essential functional items for many and come in diverse frame styles and materials.",
    "Gloves": "A covering for the hand, typically worn for protection against cold, dirt, or for specific tasks. Gloves feature separate parts for each finger and the thumb, allowing for dexterity. They come in various materials like leather, wool, knit, or rubber, serving functional or fashion purposes.",
    "Handbag": "A small to medium-sized bag designed to hold personal items like a wallet, phone, keys, and cosmetics, typically carried by women. Handbags come in numerous styles (tote, shoulder, crossbody, clutch) with various handles or straps, materials, and designs, serving as both a functional and fashion accessory.",
    "Knee-high socks": "Socks that extend up the leg to the knee, providing substantial coverage and warmth. Knee-high socks are often worn with boots, skirts, or shorts, offering various styles from athletic to fashion-forward, in a range of materials and patterns.",
    "Mittens": "A type of hand covering that encloses all four fingers together, with a separate section only for the thumb. Mittens are generally warmer than gloves because they reduce the surface area exposed to cold and allow fingers to share warmth. They are worn for protection against cold.",
    "Neck tie": "A long piece of cloth worn around the neck, beneath the collar of a shirt, and knotted at the throat, with the broad ends hanging down the front. Neckties are primarily a decorative accessory for men, typically worn with suits or formal wear, coming in various materials, patterns, and widths.",
    "Necklace": "An ornamental chain, string of beads, or pendant worn around the neck. Necklaces are a diverse category of jewelry, ranging from simple chains to elaborate statement pieces, made from various metals, gemstones, pearls, or other materials, enhancing personal style.",
    "No-show socks": "Very low-cut socks designed to be invisible when worn with shoes like sneakers, loafers, or ballet flats, giving the appearance of wearing no socks at all while still providing comfort and preventing friction. They sit just at or below the shoe line.",
    "Over-ear headphones": "Headphones characterized by large earcups that fully enclose the ears, providing superior sound isolation and often better audio quality compared to smaller headphone types. They typically connect over the top of the head with a band and are popular for immersive listening experiences.",
    "Pocket square": "A small, square piece of fabric, typically silk, linen, or cotton, designed to be folded decoratively and placed in the breast pocket of a jacket or blazer. It is a stylish accessory that adds a touch of color, pattern, or texture to formal or smart-casual attire.",
    "Ring": "A small circular band, typically made of precious metal (gold, silver, platinum) and often set with one or more gemstones, worn on a finger as an ornament. Rings hold significant cultural and symbolic meaning (e.g., wedding, engagement) and are a widely popular form of jewelry.",
    "Scarf": "A versatile length or square of fabric worn around the neck for warmth, protection from the sun, or as a fashion accessory. Scarves come in a vast array of materials (wool, silk, cotton, cashmere), sizes, patterns, and wearing styles, adaptable for various climates and outfits.",
    "Shawl": "A large piece of cloth, usually rectangular, square, or triangular, worn by women over the shoulders or head, or wrapped around the body. Shawls provide warmth, modesty, or serve as a decorative accessory, often made from fine fabrics like wool, cashmere, or silk, sometimes intricately patterned or embroidered.",
    "Shoulder bag": "A handbag with a strap long enough to be worn over one shoulder, allowing the bag to hang at the hip or waist level. Shoulder bags are a highly common and practical type of handbag, offering easy access to contents while keeping hands free. They come in countless sizes, materials, and designs.",
    "Stockings": "Long, close-fitting coverings for the legs and feet, typically made of sheer or opaque hosiery material (nylon, silk) and traditionally worn by women. Stockings extend up to the thigh and are held up by garters or elastic, worn for warmth, modesty, or fashion.",
    "Sunglasses": "Protective eyewear designed with tinted lenses to prevent bright sunlight from damaging or discomforting the eyes. Sunglasses are essential for eye protection outdoors and are also a prominent fashion accessory, coming in countless frame styles, lens colors, and materials.",
    "Tights": "A sheer, close-fitting garment, typically made of stretchy hosiery material, that covers the body from the waist to the feet. Tights are often worn by women under skirts or dresses for warmth, modesty, or aesthetic purposes, providing a smooth, continuous look.",
    "Tote bag": "A large, often unfastened bag with two parallel handles, designed for carrying a variety of items. Tote bags are typically made of sturdy cloth, canvas, or leather, known for their spacious interior and versatility, used for shopping, carrying books, beach essentials, or as everyday bags.",
    "Umbrella": "A portable device consisting of a circular canopy of cloth or plastic on a folding metal frame supported by a central rod, used as protection against rain or sometimes sun. Umbrellas come in various sizes, colors, and designs, serving as a practical accessory for inclement weather.",
    "Wallet": "A small, flat, folding case designed for carrying personal items such as cash (bills and coins), credit cards, identification documents, and sometimes photos. Wallets are an essential everyday accessory, typically made of leather or synthetic materials, carried in a pocket or bag.",
    "Wrist watch": "A timepiece worn on a strap or bracelet around the wrist. Wristwatches serve as both a functional device for telling time and a significant fashion accessory or status symbol. They come in mechanical, quartz, digital, and smart designs, with endless variations in style and features.",

    "Abaya robe": "A loose, flowing, and often simple over-garment, resembling a robe-like dress, traditionally worn by some women in parts of the Muslim world. It covers the entire body except for the face, hands, and feet, embodying modesty. Abayas come in various designs, fabrics, and embellishments, adapting to regional styles and personal preferences.",
    "Ao dai tunic": "A traditional Vietnamese national garment, typically a long, tight-fitting silk tunic with side slits extending to the waist, worn over loose-fitting trousers. It's renowned for its elegant, flowing lines that flatter the wearer's figure, commonly worn for formal occasions, ceremonies, and cultural events.",
    "Boubou robe": "A flowing, wide-sleeved robe or tunic, often brightly colored and intricately embroidered, worn by both men and women across much of West Africa and the Middle East. It is characterized by its large, rectangular shape and generous cut, offering comfort and a distinctive cultural aesthetic.",
    "Caftan robe": "A loose-fitting, flowing robe or tunic-like garment, typically with wide sleeves, that has been worn in various cultures for thousands of years. Often made from luxurious fabrics and adorned with embroidery or patterns, kaftans are known for their comfort and elegance, suitable for lounging, resort wear, or formal events.",
    "Cheongsam dress": "A body-hugging one-piece Chinese dress for women, also known as a Qipao, characterized by its body-hugging silhouette, high collar (mandarin collar), frog buttons, and side slits. Originally a modernized version of Qing Dynasty attire, it is now a popular choice for formal events, parties, and cultural celebrations, known for its elegant and sophisticated appeal.",
    "Dirndl dress": "A traditional feminine dress originating from Austria, South Tyrol, and Bavaria, based on the historical costume of Alpine peasants. It typically consists of a bodice, a blouse, a full skirt, and an apron. Dirndls are often made from natural fabrics with traditional patterns and are worn for cultural festivals, celebrations, and formal occasions in the region.",
    "Hanbok dress": "The traditional Korean dress, characterized by its vibrant colors, simple lines, and lack of pockets. For women, it typically consists of a `jeogori` (short jacket) and a `chima` (long, voluminous skirt). Hanbok is worn for holidays, festivals, and special occasions, embodying Korean cultural heritage and aesthetic.",
    "Huipil blouse": "A traditional garment worn by indigenous women in Central Mexico and Central America, typically a loose-fitting blouse or tunic, often intricately woven and embroidered with symbolic designs and vibrant colors. Each huipil can convey information about the wearer's community, marital status, or personal taste.",
    "Kente cloth top": "A top or garment made from Kente cloth, a traditional Ghanaian textile. Kente cloth is a distinctive type of silk and cotton fabric made of interwoven cloth strips, characterized by its vibrant, multicolored patterns and symbolic designs. A Kente cloth top would be a colorful and culturally significant piece of clothing.",
    "Kilt skirt": "A knee-length, pleated skirt-like garment traditionally worn by men as part of Scottish Highland dress. Kilts are made from tartan cloth, featuring specific plaid patterns that represent Scottish clans. While rooted in tradition, modern kilts or kilt-inspired skirts can also be worn as contemporary fashion.",
    "Kimono robe": "A traditional Japanese garment, a T-shaped, straight-lined robe worn wrapped around the body and secured with a sash (obi). Kimonos are typically ankle-length with long, wide sleeves. While traditional kimonos are complex formal wear, 'kimono robe' often refers to lighter, less formal versions worn as dressing gowns or stylish outerwear, known for their elegant drape and cultural aesthetic.",
    "Lederhosen pants": "Traditional German leather breeches, typically short or knee-length, worn by men as traditional garments in Bavaria and other Alpine regions. They are made from durable leather, often with distinctive embroidery and suspenders, worn for folk festivals like Oktoberfest.",
    "Sari garment": "A traditional Indian subcontinent garment, consisting of a drape of fabric (typically cotton or silk) from five to nine yards in length, wrapped around the waist, with one end draped over the shoulder, baring the midriff. It is a highly versatile and culturally significant attire.",
    "Sarong wrap": "A large tube or length of fabric, often brightly colored or patterned, typically wrapped around the waist and worn as a skirt-like garment by both men and women in the Malay Archipelago and surrounding areas. Sarongs are versatile for beachwear, casual attire, or traditional dress.",
    "Sherwani coat": "A long coat-like garment worn by men in the Indian subcontinent, very similar to a Western frock coat or a Polish żupan. Sherwanis are typically knee-length, often embroidered, and worn for formal occasions, weddings, and traditional celebrations, exuding an elegant and regal aesthetic.",
    'Faux Fur Coat': 'A luxurious, warm, and stylish coat with a fluffy, soft texture that mimics real fur. It can be long or short and comes in various colors.',
    'Dashiki tunic': 'A vibrant and colorful traditional West African garment. It is a loose-fitting tunic with intricate, embroidered patterns, typically worn by men and women.',
    "Bra": "A form-fitting top designed to support and cover the breasts, typically featuring cups, straps, and a back closure. Bras are made for everyday wear and come in various styles like sports bras, push-up bras, and bralettes.",
    "Panty": "A type of underwear worn by women and girls, covering the pelvic area. They are often made from soft, breathable materials and come in a wide range of styles, including briefs, thongs, and bikinis. They are designed for comfort and hygiene.",
    "Boxers": "Loose-fitting underwear or shorts typically worn by men. They have an elastic waistband and fall to mid-thigh, providing comfort and breathability. They can be worn as underwear or as loungewear.",
    "One-piece swimsuit": "A single garment that covers the torso, often used for swimming. Unlike a bikini, it does not have separate top and bottom pieces. One-piece swimsuits are designed for functionality and style in water activities."

}

CATEGORY_LIST_BY_GROUP = {
    "Tops": [
        "T-shirt", "Polo shirt", "Jersey", "Button-down shirt", "Henley shirt",
        "Tank top", "Knit sweater", "Blouse", "Tunic", "Crop top",
        "Sleeveless top", "Pullover hoodie", "Turtleneck", "button-up shirt",
        "Dashiki tunic", "Ao dai tunic", "Huipil blouse", "Kente cloth top"
    ],
    "Outerwear": [
        "Denim jacket", "Leather jacket", "Puffer jacket", "Trench coat",
        "Peacoat", "Blazer", "Windbreaker jacket", "Cardigan sweater",
        "Vest", "Raincoat", "Parka", "Zippered hoodie", "Tracksuit jacket",
        "Coat", "Jacket", "Leather bomber jacket", "Leather coat", "Faux Fur Coat",
        "Kimono robe", "Sherwani coat"
    ],
    "Bottoms": [
        "Straight-leg jeans", "Skinny jeans", "Bootcut jeans", "Cargo pants",
        "Chino pants", "Dress pants", "Shorts", "Capri pants", "Leggings",
        "Joggers", "Denim shorts", "Sweatpants", "Trousers",
        "Lederhosen pants"
    ],
    "Skirts": [
        "A-line skirt", "Pencil skirt", "Maxi skirt", "Mini skirt",
        "Pleated skirt", "Wrap skirt", "Denim skirt",
        "Kilt skirt", "Sarong wrap"
    ],
    "Dresses & Rompers": [
        "Strap dress", "Wrap dress", "T-shirt dress", "Maxi dress", "Dress",
        "Midi dress", "Mini dress", "Cocktail dress", "Evening gown",
        "Sundress", "Shirt dress", "Sweater dress", "Overalls",
        "Jumpsuit", "Romper", "Denim dress",
        "Sari garment", "Hanbok dress", "Dirndl dress", "Cheongsam dress",
        "Boubou robe", "Caftan robe", "Abaya robe","Dress"
    ],
    "Footwear": [
        "Sneakers", "Oxford dress shoes", "Ankle boots", "Knee-high boots",
        "Heel pumps", "Sandals", "Rain boots", "Loafers", "Ballet flats",
        "Wedges", "Espadrilles", "Slippers", "Suede dress shoes", "Dress shoe"
    ],
    "Headwear": [
        "Baseball cap", "Beanie hat", "Bucket hat", "Sun hat", "Headband",
        "Headscarf", "Hijab head covering", "Beret", "Fedora", "Fez hat",
        "Turban", "Sombrero"
    ],
    "Bags": [
        "Handbag", "Backpack", "Tote bag", "Clutch bag", "Shoulder bag",
        "Crossbody bag", "Wallet"
    ],
    "Neckwear": [
        "Neck tie", "Bow tie", "Scarf", "Shawl", "Bandana", "Necklace"
    ],
    "Earwear": [
        "Earrings", "Over-ear headphones", "Earbuds"
    ],
    "Wristwear": [
        "Wrist watch", "Bracelet"
    ],
    "Handwear": [
        "Gloves", "Mittens"
    ],
    "Eyewear": [
        "Sunglasses", "Eyeglasses"
    ],
    "Socks & Hosiery": [
        "Knee-high socks", "Ankle socks", "Crew socks", "No-show socks",
        "Dress socks", "Stockings", "Tights"
    ],
    "Other Accessories": [
        "Belt", "Ring", "Brooch", "Pocket square", "Umbrella"
    ],
    "Underwear & Swimwear": [
        "Bra", "Panty", "One-piece swimsuit", "Boxers"
    ]
}

PATTERN_DESCRIPTIONS = {
    "solid": "A single, uniform color with no pattern or print.",
    "striped": "A pattern of parallel lines or bands of different colors.",
    "checked": "A pattern of intersecting horizontal and vertical lines forming squares, like a checkerboard. Also known as checker.",
    "plaid": "A pattern of intersecting horizontal and vertical bands in multiple colors. Tartan is a specific type of plaid.",
    "floral": "A pattern of floral print featuring flowers, leaves, and other botanical elements.",
    "polka dots": "A pattern consisting of filled circles of the same size.",
    "geometric": "A pattern made of geometric shapes like triangles, circles, squares, or lines.",
    "paisley": "A distinctive intricate pattern of curved, feather-shaped figures based on a pine-cone design from India.",
    "animal print": "A pattern that imitates the skin or fur of an animal, such as leopard, zebra, or snake.",
    "tie-dye": "A pattern created by tying sections of fabric before dyeing to create irregular, colorful designs.",
    "camouflage": "A pattern of mottled colors, typically greens and browns, used to blend in with the surroundings.",
    "ombre": "A pattern with a gradual blending of one color hue to another, usually moving tints and shades.",
    "color-block": "A pattern using two or more large, solid blocks of color on a single garment.",
    "jacquard": "An intricate, textured pattern that is woven directly into the fabric, rather than printed on top.",
    "houndstooth": "A two-tone pattern of broken checks or abstract four-pointed shapes.",
    "batik": "A pattern created using a wax-resist dyeing technique, resulting in intricate, fluid designs.",
    "graphic": "A printed or embroidered design featuring logo, brand names, slogans, text in various fonts (cursive, block, capital letters), words or phrases, numbers, icons, illustrations, or artwork, covering part or the entire garment.",
    "textured": "A surface with raised, three-dimensional details such as ruffles, embellishments, or fabric manipulation, creating a tactile feel rather than a printed or woven pattern.",
}

CLIP_PATTERN_DESCRIPTIONS = {
    "solid": "A single, unbroken color with no design, no print, or no visible pattern. The fabric appears uniform throughout.",
    "striped": "Repeating straight lines running across the fabric.",
    "checked": "Squares made by crossing lines, like a checkerboard.",
    "plaid": "Overlapping lines of different colors forming a grid.",
    "floral": "Repeating images of flowers, leaves, or a plant-based design.",
    "geometric": "A design made of abstract or regular shapes like triangles, circles, or polygons.",
    "lace": "A patterned fabric made with a series of connected threads, often with floral or intricate designs.",
    "textured": "A raised or 3D surface you can feel, created by knitting, weaving, or quilting.",
    "polka dots": "Evenly spaced, same-size circles on the fabric.",
    "paisley": "Curved, teardrop-shaped figures, often detailed.",
    "animal print": "Spots or stripes that look like animal skin or fur.",
    "tie-dye": "Swirled or blotchy areas of different colors.",
    "camouflage": "camouflage, irregular shapes in earth tones, for blending in.",
    "ombre": "Color that fades smoothly from light to dark or between colors.",
    "color-block": "Large, solid blocks of different colors.",
    "jacquard": "Raised, woven-in patterns with texture, not printed.",
    "houndstooth": "Jagged, duotone textile,  abstract shapes that look like dog’s teeth.",
    "batik": "Blurry, flowing designs with a hand-dyed look.",
    "graphic": "graphic artwork , logos, or text on the fabric.",
    "cable knit": "Textured knit fabric with raised, twisted cable ."
}

CLIP_MATERIAL_DESCRIPTIONS = {
    "cotton": "Natural fiber fabric, soft, breathable, and comfortable for everyday wear.",
    "linen": "Fabric made only from flax plant fibers.",
    "silk": "Fabric made only from silkworm fibers.",
    "wool": "Fabric made only from sheep fibers.",
    "leather": "Material made only from animal hide.",
    "denim": "Sturdy fabric , classic for jeans and jackets.",
    "polyester": "Fabric made only from polyester fibers.",
    "nylon": "Fabric made only from nylon fibers.",
    "spandex": "Fabric made only from elastic spandex fibers.",
    "knit": "Fabric made by interlocking loops of yarn.",
    "velvet": "Fabric with a short, dense pile on the surface.",
    "patent leather": "Leather with a shiny, smooth surface.",
    "suede": "Leather with a soft, napped surface.",
    "chiffon": "Lightweight, sheer, plain-woven fabric.",
    "mesh": "Fabric with an open, net-like structure.",
    "canvas": "Heavy-duty, plain-woven fabric.",
    "faux leather": "Artificial material made to look like leather.",
    "faux fur": "Artificial material made to look like animal fur.",
    "fleece": "Synthetic insulating fabric with a soft, napped surface.",
    "quilted": "Fabric with two layers stitched together in a pattern.",
    "blend": "Fabric made from two or more different types of fibers mixed together.",
    "cashmere": "Luxurious, soft wool from cashmere goats, known for its warmth and silky texture.",
    "water proof": "A material that prevents water from passing through",
}

FULL_CLIP_MATERIAL_DESCRIPTIONS = {
    "canvas": "A heavy-duty, plain-woven fabric that is stiff, rugged, and durable.",
    "cashmere": "A super-soft, luxurious, and lightweight wool from cashmere goats with a silky feel and exceptional warmth.",
    "chiffon": "An extremely lightweight, sheer, and airy fabric that drapes with a delicate, flowing movement.",
    "cotton": "A soft, fluffy, natural fabric that feels cool and breathable. It often has a matte finish and is used for t-shirts and casual clothing.",
    "denim": "A rugged, thick, and durable cotton twill fabric with a characteristic diagonal weave. It's commonly known for its deep blue color in jeans.",
    "faux fur": "A synthetic material made of long, fibrous strands that closely resembles the fluffy texture of animal fur.",
    "faux leather": "An artificial, man-made material designed to mimic the grainy appearance and texture of real leather, often with a smooth, plastic-like feel.",
    "fleece": "A soft, fuzzy synthetic fabric with a deep pile that feels warm and insulating, often used in jackets and blankets.",
    "gold": "A precious, shiny, yellow metal used in jewelry and as a decorative element on clothing and accessories.",
    "knit": "A stretchy fabric created by interlocking loops of yarn, giving it a noticeable, textured surface and natural pliability.",
    "lace": "A delicate, intricately woven or knotted fabric with open spaces, creating a sheer and ornate pattern often used for decoration.",
    "leather": "A stiff, durable material made from tanned animal hide. It has a smooth, often grainy surface and can be polished to a shine.",
    "linen": "A crisp, natural fabric with a slightly rough, textured feel. It wrinkles easily and is known for its light, airy drape.",
    "mesh": "An open, net-like fabric with visible holes or a grid structure, creating a breathable, perforated look.",
    "nylon": "A very strong and lightweight synthetic fabric that feels slick and smooth. It's known for its high durability and water resistance.",
    "patent leather": "A type of leather with a highly reflective, glossy finish that makes it look like a polished, solid surface.",
    "plastic": "A synthetic material that can be molded into solid objects, used for buttons, zippers, and sometimes as a transparent material in raincoats or bags.",
    "polyester": "A man-made, durable synthetic fabric that feels slick or smooth. It's resistant to wrinkles, shrinking, and stretching.",
    "quilted": "Fabric created by stitching two layers together in a pattern, forming a puffy, raised, and insulated texture.",
    "satin": "A fabric with a very smooth, lustrous, and reflective front surface and a dull back, known for its elegant, draping quality.",
    "silk": "A luxurious, smooth, and lightweight fabric with a distinctive, shimmering sheen. It feels very soft and cool to the touch.",
    "silver": "A precious, shiny, grayish-white metal used in jewelry and as a decorative element.",
    "spandex": "An extremely elastic synthetic fiber known for its high stretch and recovery. It feels very smooth and form-fitting.",
    "suede": "A soft, velvety leather with a brushed, napped surface that feels fuzzy and luxurious.",
    "velvet": "A soft, plush fabric with a dense, raised pile on the surface that gives it a rich, luxurious feel and a subtle sheen.",
    "wool": "A thick, warm fabric made from animal fleece, often with a coarse, fuzzy texture. It can be woven into heavy coats or soft sweaters.",
    "blend": "A fabric composed of two or more different fibers woven together, combining the characteristics of each, such as a soft cotton-polyester mix.",
    "water proof": "A material that prevents water from passing through",
}


CATEGORY_DEFAULT_MATERIAL = {

    "T-shirt":              "cotton",
    "Polo shirt":           "cotton",
    "Jersey":               "polyester",
    "Button-down shirt":    "cotton",
    "Henley shirt":         "cotton",
    "Tank top":             "cotton",
    "Knit sweater":         "wool",
    "Blouse":               "silk",
    "Tunic":                "cotton",
    "Crop top":             "cotton",
    "Sleeveless top":       "cotton",
    "Pullover hoodie":      "cotton",
    "Turtleneck":           "wool",


    "Denim jacket":         "denim",
    "Leather jacket":       "leather",
    "Quilted jacket":        "polyester",
    "Trench coat":          "wool",
    "Peacoat":              "wool",
    "Blazer":               "wool",
    "Coat":                  "wool",
    "Windbreaker jacket":   "nylon",
    "Cardigan sweater":     "wool",
    "Vest":                 "polyester",
    "Raincoat":             "polyester",
    "Parka":                "down",
    "Zippered hoodie":      "cotton",
    "Tracksuit jacket":     "polyester",
    "Jacket":               "water proof",
    "Duster coat"     :     "leather",
    "Leather bomber jacket":"leather",



    "Straight-leg jeans":   "denim",
    "Skinny jeans":         "denim",
    "Bootcut jeans":        "denim",
    "Cargo pants":          "cotton",
    "Chino pants":          "cotton",
    "Dress pants":          "wool",
    "Shorts":               "cotton",
    "Capri pants":          "cotton",
    "Leggings":             "spandex",
    "Joggers":              "polyester",
    "Sweatpants":          "cotton",
    "Trousers":             "cotton",



    "A-line skirt":         "cotton",
    "Pencil skirt":         "wool",
    "Maxi skirt":           "cotton",
    "Mini skirt":           "cotton",
    "Pleated skirt":        "polyester",
    "Wrap skirt":           "cotton",
    "Denim skirt":          "denim",


    "Strap dress":          "cotton",
    "Wrap dress":           "cotton",
    "T-shirt dress":        "cotton",
    "Maxi dress":           "cotton",
    "Midi dress":           "cotton",
    "Mini dress":           "cotton",
    "Cocktail dress":       "silk",
    "Evening gown":         "silk",
    "Sundress":             "cotton",
    "Shirt dress":          "cotton",
    "Sweater dress":        "wool",
    "Overalls":             "denim",
    "Jumpsuit":             "polyester",
    "Romper":               "cotton",
   "Denim dress":           "denim",


    "Sneakers":             "synthetic",
    "Oxford dress shoes":   "leather",
    "Ankle boots":          "leather",
    "Knee-high boots":      "leather",
    "Heel pumps":           "leather",
    "Sandals":              "leather",
    "Rain boots":           "rubber",
    "Loafers":              "leather",
    "Ballet flats":         "leather",
    "Wedges":               "leather",
    "Espadrilles":          "canvas",
    "Slippers":             "faux fur",
    "Suede dress shoes":    "suede",
    "Dress shoe":           "leather",


    "Baseball cap":         "cotton",
    "Beanie hat":           "wool",
    "Bucket hat":           "cotton",
    "Sun hat":              "straw",
    "Headband":             "cotton",
    "Headscarf":            "silk",
    "Hijab head covering":  "cotton",
    "Beret":                "wool",
    "Fedora":               "felt",
    "Fez hat":              "felt",
    "Turban":               "cotton",
    "Sombrero":             "straw",


    "Handbag":              "leather",
    "Backpack":             "nylon",
    "Tote bag":             "canvas",
    "Clutch bag":           "leather",
    "Shoulder bag":         "leather",
    "Crossbody bag":        "leather",
    "Wallet":               "leather",
    "Belt":                 "leather",
    "Neck tie":             "silk",
    "Bow tie":              "silk",
    "Bandana":              "cotton",
    "Gloves":               "leather",
    "Mittens":              "wool",
    "Knee-high socks":      "cotton",
    "Ankle socks":          "cotton",
    "Crew socks":           "cotton",
    "No-show socks":        "cotton",
    "Dress socks":          "wool",
    "Stockings":            "nylon",
    "Tights":               "nylon",
    "Wrist watch":          "metal",
    "Necklace":             "metal",
    "Earrings":             "metal",
    "Bracelet":             "metal",
    "Ring":                 "metal",
    "Brooch":               "metal",
    "Pocket square":        "silk",
    "Over-ear headphones":  "plastic",
    "Earbuds":              "plastic",
    "Sunglasses":           "plastic",
    "Eyeglasses":           "plastic",
    "Scarf":                "silk",
    "Shawl":                "wool",
    "Umbrella":             "metal",


    "Kimono robe":          "silk",
    "Sari garment":         "silk",
    "Kilt skirt":           "wool",
    "Dashiki tunic":        "cotton",
    "Sherwani coat":        "silk",
    "Hanbok dress":         "silk",
    "Dirndl dress":         "wool",
    "Lederhosen pants":     "leather",
    "Ao dai tunic":         "silk",
    "Cheongsam dress":      "silk",
    "Boubou robe":          "cotton",
    "Caftan robe":          "silk",
    "Huipil blouse":        "cotton",
    "Kente cloth top":      "cotton",
    "Abaya robe":           "cotton",
    "Sarong wrap":          "cotton",
}


CATEGORY_DEFAULT_LENGTH= {

    "T-shirt": "standard",
    "Polo shirt": "standard",
    "Jersey": "standard",
    "Button-down shirt": "standard",
    "Henley shirt": "standard",
    "Tank top": "standard",
    "Knit sweater": "standard",
    "Blouse": "standard",
    "Tunic": "tunic",
    "Crop top": "crop",
    "Sleeveless top": "standard",
    "Pullover hoodie": "standard",
    "Turtleneck": "standard",


    "Denim jacket": "hip",
    "Leather jacket": "hip",
    "Puffer jacket": "hip",
    "Trench coat": "knee",
    "Peacoat": "thigh",
    "Blazer": "hip",
    "Coat": "knee",
    "Windbreaker jacket": "hip",
    "Cardigan sweater": "standard",
    "Vest": "hip",
    "Raincoat": "knee",
    "Parka": "thigh",
    "Zippered hoodie": "standard",
    "Tracksuit jacket": "hip",
    "Jacket": "hip",


    "Straight-leg jeans": "ankle",
    "Skinny jeans": "ankle",
    "Bootcut jeans": "full",
    "Cargo pants": "ankle",
    "Chino pants": "ankle",
    "Dress pants": "ankle",
    "Shorts": "short",
    "Capri pants": "capri",
    "Leggings": "ankle",
    "Joggers": "ankle",
    "Denim shorts": "short",


    "A-line skirt": "knee",
    "Pencil skirt": "knee",
    "Maxi skirt": "maxi",
    "Mini skirt": "mini",
    "Pleated skirt": "knee",
    "Wrap skirt": "knee",
    "Denim skirt": "knee",


    "Strap dress": "mini",
    "Wrap dress": "knee",
    "T-shirt dress": "knee",
    "Maxi dress": "maxi",
    "Midi dress": "midi",
    "Mini dress": "mini",
    "Cocktail dress": "knee",
    "Evening gown": "maxi",
    "Sundress": "knee",
    "Shirt dress": "knee",
    "Sweater dress": "knee",
    "Overalls": "ankle",
    "Jumpsuit": "ankle",
    "Romper": "short",


    "Sneakers": "standard",
    "Oxford dress shoes": "standard",
    "Ankle boots": "ankle",
    "Knee-high boots": "knee",
    "Heel pumps": "standard",
    "Sandals": "standard",
    "Rain boots": "standard",
    "Loafers": "standard",
    "Ballet flats": "standard",
    "Wedges": "standard",
    "Espadrilles": "standard",
    "Slippers": "standard",
    "Suede dress shoes": "standard",
    "Dress shoe": "standard",



    "Kimono robe": "maxi",
    "Sari garment": "maxi",
    "Kilt skirt": "knee",
    "Dashiki tunic": "standard",
    "Sherwani coat": "knee",
    "Hanbok dress": "midi",
    "Dirndl dress": "knee",
    "Lederhosen pants": "short",
    "Ao dai tunic": "maxi",
    "Cheongsam dress": "knee",
    "Boubou robe": "maxi",
    "Caftan robe": "maxi",
    "Huipil blouse": "standard",
    "Kente cloth top": "standard",
    "Abaya robe": "maxi",
    "Sarong wrap": "knee"
}

COLOR_GROUPS = {
    "neutrals": [
        "black", "white", "grey", "gray", "navy", "beige", "taupe", "tan", "brown", "ivory",
        "cream", "off-white", "charcoal", "stone", "khaki"
    ],
    "pastels": [
        "light", "pale", "pastel", "soft", "powder", "mint", "baby", "blush",
        "pastel pink", "pastel blue", "pastel green", "lavender", "peach", "sky blue", "mint green", "lemon", "lilac"
    ],
    "brights": [
        "bright", "yellow", "red", "blue", "green", "orange", "purple", "pink",
        "vibrant", "neon", "electric", "bold", "hot", "true",
        "turquoise", "fuchsia", "lime", "cyan", "magenta"
    ],
    "darks": [
        "dark", "deep", "midnight", "burgundy", "forest", "charcoal", "oxblood", "espresso",
        "dark blue", "dark green", "dark red", "maroon", "olive","aubergine"
    ],
    "metallics": [
        "gold", "silver", "bronze", "copper", "metallic", "rose gold"
    ]
}

COLOR_GROUPS_MAP = {
    "neutrals": [
        "black", "white", "grey", "gray", "navy", "beige", "taupe", "tan", "brown", "ivory",
        "cream", "off-white", "charcoal", "stone", "khaki"
    ],
    "pastels": [
        "light", "pale", "pastel", "soft", "powder", "mint", "baby", "blush",
        "pastel pink", "pastel blue", "pastel green", "lavender", "peach", "sky blue", "mint green", "lemon", "lilac"
    ],
    "brights": [
        "bright", "yellow", "red", "blue", "green", "orange", "purple", "pink",
        "vibrant", "neon", "electric", "bold", "hot", "true",
        "turquoise", "fuchsia", "lime", "cyan", "magenta"
    ],
    "darks": [
        "dark", "deep", "midnight", "burgundy", "forest", "charcoal", "oxblood", "espresso",
        "dark blue", "dark green", "dark red", "maroon", "olive","aubergine"
    ],
    "metallics": [
        "gold", "silver", "bronze", "copper", "metallic", "rose gold"
    ]
}

KEYWORD_TO_CATEGORY = {
    # Tops
    "blouse": "Blouse", "blouses": "Blouse", "top": "Blouse", "peasant blouse": "Blouse", "smock": "Blouse",
    "button-down shirt": "Button-down shirt", "button down": "Button-down shirt", "collared shirt": "Button-down shirt", "dress shirt": "Button-down shirt", "oxford shirt": "Button-down shirt",
    "button-up shirt": "Button-up shirt",
    "crop top": "Crop top", "cropped top": "Crop top", "belly shirt": "Crop top", "short top": "Crop top",
    "henley shirt": "Henley shirt", "henley": "Henley shirt", "placket shirt": "Henley shirt",
    "jersey": "Jersey", "sports jersey": "Jersey", "athletic jersey": "Jersey", "uniform jersey": "Jersey",
    "knit sweater": "Knit sweater", "sweater": "Knit sweater", "knitted top": "Knit sweater", "jumper": "Knit sweater", "pullover": "Knit sweater", "crewneck sweater": "Knit sweater", "cardigan": "Cardigan sweater", "cardigans": "Cardigan sweater",
    "polo shirt": "Polo shirt", "polo": "Polo shirt", "golf shirt": "Polo shirt", "lacoste shirt": "Polo shirt",
    "pullover hoodie": "Pullover hoodie", "hoodie": "Pullover hoodie", "hooded sweatshirt": "Pullover hoodie", "hooded sweater": "Pullover hoodie", "jumper with hood": "Pullover hoodie",
    "sleeveless top": "Sleeveless top", "sleeveless blouse": "Sleeveless top", "cut-off top": "Sleeveless top",
    "tank top": "Tank top", "tank": "Tank top", "cami": "Tank top", "camisole": "Tank top", "sleeveless shirt": "Tank top", "undershirt": "Tank top",
    "t-shirt": "T-shirt", "tee shirt": "T-shirt", "tee": "T-shirt", "crewneck t-shirt": "T-shirt", "v-neck t-shirt": "T-shirt", "graphic tee": "T-shirt",
    "tunic": "Tunic", "tunic top": "Tunic", "long shirt": "Tunic",
    "turtleneck": "Turtleneck", "roll neck": "Turtleneck", "high-neck top": "Turtleneck",

    # Outerwear
    "blazer": "Blazer", "suit jacket": "Blazer", "sport coat": "Blazer", "dress jacket": "Blazer",
    "cardigan sweater": "Cardigan sweater", "cardigan": "Cardigan sweater", "knit jacket": "Cardigan sweater", "wrap cardigan": "Cardigan sweater",
    "denim jacket": "Denim jacket", "jean jacket": "Denim jacket", "trucker jacket": "Denim jacket",
    "leather jacket": "Leather jacket", "biker jacket": "Leather jacket", "moto jacket": "Leather jacket", "leather coat": "Leather coat",
    "parka": "Parka", "anorak": "Parka", "winter coat": "Parka", "heavy coat": "Parka",
    "peacoat": "Peacoat", "pea coat": "Peacoat", "winter jacket": "Peacoat", "wool coat": "Peacoat",
    "puffer jacket": "Puffer jacket", "puffer coat": "Puffer jacket", "down jacket": "Puffer jacket", "quilted jacket": "Puffer jacket",
    "raincoat": "Raincoat", "rain jacket": "Raincoat", "slicker": "Raincoat", "waterproof coat": "Raincoat",
    "trench coat": "Trench coat", "trench": "Trench coat", "overcoat": "Coat", "long coat": "Trench coat",
    "vest": "Vest", "waistcoat": "Vest", "gilet": "Vest",
    "windbreaker jacket": "Windbreaker jacket", "windbreaker": "Windbreaker jacket", "wind jacket": "Windbreaker jacket",
    "zippered hoodie": "Zippered hoodie", "zip-up hoodie": "Zippered hoodie",
    "tracksuit jacket": "Tracksuit jacket", "track jacket": "Tracksuit jacket",
    "leather bomber jacket": "Leather bomber jacket", "bomber jacket": "Leather bomber jacket", "flight jacket": "Leather bomber jacket",
    "coat": "Coat", "overcoat": "Coat",
    "jacket": "Jacket",

    # Pants & Shorts
    "bootcut jeans": "Bootcut jeans",
    "capri pants": "Capri pants", "capris": "Capri pants", "cropped pants": "Capri pants", "three-quarter pants": "Capri pants",
    "cargo pants": "Cargo pants", "cargo trousers": "Cargo pants", "utility pants": "Cargo pants",
    "chino pants": "Chino pants", "chinos": "Chino pants", "khaki pants": "Chino pants",
    "sweatpants": "Sweatpants", "sweats": "Sweatpants", "track pants": "Sweatpants", "fleece pants": "Sweatpants",
    "dress pants": "Dress pants", "dress trousers": "Dress pants", "formal pants": "Dress pants",
    "joggers": "Joggers", "jogging pants": "Joggers",
    "leggings": "Leggings", "tights": "Leggings", "yoga pants": "Leggings",
    "shorts": "Shorts", "short pants": "Shorts",
    "skinny jeans": "Skinny jeans", "slim-fit jeans": "Skinny jeans", "skinnies": "Skinny jeans",
    "straight-leg jeans": "Straight-leg jeans", "jeans": "Straight-leg jeans", "denim pants": "Straight-leg jeans",
    "trousers": "Trousers", "pants": "Trousers",
    "denim shorts": "Denim shorts", "jean shorts": "Denim shorts",

    # Skirts
    "a-line skirt": "A-line skirt", "flared skirt": "A-line skirt", "fit-and-flare skirt": "A-line skirt",
    "denim skirt": "Denim skirt", "jean skirt": "Denim skirt",
    "maxi skirt": "Maxi skirt", "long skirt": "Maxi skirt", "floor-length skirt": "Maxi skirt",
    "mini skirt": "Mini skirt", "short skirt": "Mini skirt", "micro skirt": "Mini skirt",
    "pencil skirt": "Pencil skirt", "tube skirt": "Pencil skirt",
    "pleated skirt": "Pleated skirt", "pleat skirt": "Pleated skirt",
    "wrap skirt": "Wrap skirt", "sarong skirt": "Wrap skirt",

    # Dresses & Jumpsuits
    "cocktail dress": "Cocktail dress", "party dress": "Cocktail dress",
    "evening gown": "Evening gown", "ball gown": "Evening gown", "formal dress": "Evening gown",
    "jumpsuit": "Jumpsuit", "one-piece suit": "Jumpsuit", "romper": "Romper", "playsuit": "Romper",
    "maxi dress": "Maxi dress", "long dress": "Maxi dress", "floor-length dress": "Maxi dress",
    "midi dress": "Midi dress", "mid-length dress": "Midi dress",
    "mini dress": "Mini dress", "short dress": "Mini dress",
    "overalls": "Overalls", "dungarees": "Overalls",
    "shirt dress": "Shirt dress", "button-down dress": "Shirt dress",
    "strap dress": "Strap dress", "spaghetti strap dress": "Strap dress",
    "sundress": "Sundress", "summer dress": "Sundress",
    "sweater dress": "Sweater dress", "knit dress": "Sweater dress",
    "t-shirt dress": "T-shirt dress",
    "wrap dress": "Wrap dress",
    "denim dress": "Denim dress", "jean dress": "Denim dress",
    "dress": "Dress", "dresses": "Dress",

    # Shoes
    "ankle boots": "Ankle boots", "boots": "Ankle boots", "booties": "Ankle boots",
    "ballet flats": "Ballet flats", "flats": "Ballet flats", "flat shoes": "Ballet flats",
    "espadrilles": "Espadrilles",
    "heel pumps": "Heel pumps", "pumps": "Heel pumps", "high heels": "Heel pumps", "stiletto": "Heel pumps",
    "knee-high boots": "Knee-high boots", "tall boots": "Knee-high boots",
    "loafers": "Loafers", "slip-on shoes": "Loafers",
    "oxford dress shoes": "Oxford dress shoes", "oxfords": "Oxford dress shoes", "dress shoes": "Oxford dress shoes", "brogues": "Oxford dress shoes",
    "rain boots": "Rain boots", "wellingtons": "Rain boots", "gum boots": "Rain boots",
    "sandals": "Sandals", "flip flops": "Sandals", "slides": "Sandals", "slip-ons": "Sandals",
    "sneakers": "Sneakers", "running shoes": "Sneakers", "trainers": "Sneakers", "tennis shoes": "Sneakers", "athletic shoes": "Sneakers", "kicks": "Sneakers",
    "slippers": "Slippers", "house shoes": "Slippers",
    "wedges": "Wedges",
    "suede dress shoes": "Suede dress shoes",
    "dress shoe": "Dress shoe",

    # Headwear
    "baseball cap": "Baseball cap", "cap": "Baseball cap", "sports cap": "Baseball cap",
    "beanie hat": "Beanie hat", "beanie": "Beanie hat", "knit cap": "Beanie hat", "skull cap": "Beanie hat",
    "beret": "Beret",
    "bucket hat": "Bucket hat",
    "fedora": "Fedora",
    "fez hat": "Fez hat", "fez": "Fez hat",
    "headband": "Headband", "hair band": "Headband",
    "headscarf": "Headscarf", "scarf": "Headscarf",
    "hijab head covering": "Hijab head covering", "hijab": "Hijab head covering",
    "sombrero": "Sombrero",
    "sun hat": "Sun hat", "wide brim hat": "Sun hat", "beach hat": "Sun hat",
    "turban": "Turban",

    # Accessories
    "ankle socks": "Ankle socks", "short socks": "Ankle socks", "socks": "Ankle socks", "sock": "Ankle socks",
    "backpack": "Backpack", "bookbag": "Backpack", "school bag": "Backpack",
    "bandana": "Bandana", "headbandana": "Bandana",
    "belt": "Belt", "waist belt": "Belt",
    "bow tie": "Bow tie",
    "bracelet": "Bracelet", "cuff": "Bracelet",
    "brooch": "Brooch",
    "clutch bag": "Clutch bag", "clutch": "Clutch bag", "evening bag": "Clutch bag",
    "crew socks": "Crew socks",
    "crossbody bag": "Crossbody bag", "crossbody": "Crossbody bag", "messenger bag": "Crossbody bag",
    "dress socks": "Dress socks",
    "earbuds": "Earbuds", "in-ear headphones": "Earbuds",
    "earrings": "Earrings", "studs": "Earrings", "hoops": "Earrings",
    "eyeglasses": "Eyeglasses", "glasses": "Eyeglasses", "spectacles": "Eyeglasses",
    "gloves": "Gloves", "mittens": "Mittens",
    "handbag": "Handbag", "purse": "Handbag", "satchel": "Handbag",
    "knee-high socks": "Knee-high socks", "tall socks": "Knee-high socks",
    "neck tie": "Neck tie", "tie": "Neck tie",
    "necklace": "Necklace", "pendant": "Necklace", "chain": "Necklace",
    "no-show socks": "No-show socks", "invisible socks": "No-show socks",
    "over-ear headphones": "Over-ear headphones", "headphones": "Over-ear headphones", "headset": "Over-ear headphones",
    "pocket square": "Pocket square", "handkerchief": "Pocket square",
    "ring": "Ring", "wedding band": "Ring",
    "scarf": "Scarf", "shawl": "Shawl", "cowl": "Scarf",
    "shoulder bag": "Shoulder bag", "hobo bag": "Shoulder bag",
    "stockings": "Stockings", "nylons": "Stockings",
    "sunglasses": "Sunglasses", "shades": "Sunglasses",
    "tights": "Tights", "pantyhose": "Tights",
    "tote bag": "Tote bag", "tote": "Tote bag", "shopper bag": "Tote bag",
    "umbrella": "Umbrella",
    "wallet": "Wallet", "money clip": "Wallet",
    "wrist watch": "Wrist watch", "watch": "Wrist watch",

    # Traditional & Underwear
    "abaya robe": "Abaya robe", "abaya": "Abaya robe",
    "ao dai tunic": "Ao dai tunic", "ao dai": "Ao dai tunic",
    "boubou robe": "Boubou robe", "boubou": "Boubou robe",
    "caftan robe": "Caftan robe", "kaftan": "Caftan robe",
    "cheongsam dress": "Cheongsam dress", "qipao": "Cheongsam dress",
    "dirndl dress": "Dirndl dress", "dirndl": "Dirndl dress",
    "hanbok dress": "Hanbok dress", "hanbok": "Hanbok dress",
    "huipil blouse": "Huipil blouse", "huipil": "Huipil blouse",
    "kente cloth top": "Kente cloth top", "kente cloth": "Kente cloth top",
    "kilt skirt": "Kilt skirt", "kilt": "Kilt skirt",
    "kimono robe": "Kimono robe", "kimono": "Kimono robe",
    "lederhosen pants": "Lederhosen pants", "lederhosen": "Lederhosen pants",
    "poncho": "Poncho",
    "sari garment": "Sari garment", "sari": "Sari garment",
    "sarong wrap": "Sarong wrap", "sarong": "Sarong wrap",
    "sherwani coat": "Sherwani coat", "sherwani": "Sherwani coat",
    "bra": "Bra", "brassiere": "Bra", "bralette": "Bra", "sports bra": "Bra", "underwire bra": "Bra",
    "panty": "Panty", "panties": "Panty", "underwear": "Panty", "undies": "Panty", "bikini briefs": "Panty", "thong": "Panty", "g-string": "Panty", "briefs": "Panty", "boy shorts": "Panty",
    "boxers": "Boxers", "boxer shorts": "Boxers", "boxer briefs": "Boxers",
    "one-piece swimsuit": "One-piece swimsuit", "swimsuit": "One-piece swimsuit", "bathing suit": "One-piece swimsuit", "one-piece bathing suit": "One-piece swimsuit",
}

ATTRIBUTES = {
    "Tops": {
        "sleeve": [
            "sleeveless (no sleeves, exposing the arms)",
            "short (sleeves ending above the elbow)",
            "long (sleeves extending to the wrist)"
        ],
        "neckline": [
            "crew (round neckline that sits close to the base of the neck)",
            "v-neck (neckline shaped like the letter 'V')",
            "scoop (wide, rounded neckline that dips lower than a crew neck)",
            "boat (wide, shallow neckline that runs horizontally across the collarbone)",
            "collared (features a traditional shirt collar)",
            "turtleneck (high, close-fitting collar that covers the neck)"
        ],
        "fit": [
            "slim",
            "regular",
            "relaxed",
            "oversized"
        ],
        "length": [
            "crop (top ends noticeably above the natural waist, exposing the midriff)",
            "standard (garment falls around the hip bone at the natural waistline)",
            "tunic (extends past the hips, often reaching mid-thigh or lower)"
        ],
        "closure": [
            "pullover (designed to be pulled over the head or up the legs, no opening)",
            "zipper (fastens with a zipper)",
            "button (fastens with buttons)",
            "tie (fastens by tying fabric or strings)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": list(CLIP_MATERIAL_DESCRIPTIONS.keys())
    },
    "Outerwear": {
        "sleeve": [
            "sleeveless (no sleeves, completely exposing the arms from the shoulder)",
             "short (sleeves ending above the elbow, typically covering only the upper arm)",
             "long (sleeves extending fully to the wrist, providing complete arm coverage)",

        ],
        "style": [
            "blazer (a tailored jacket, often with lapels, worn as formal or smart-casual outerwear)",
            "bomber (a short, waist-length jacket with a fitted waistband and cuffs, often with a zip front)",
            "parka (a long, insulated coat with a hood, designed for cold weather)",
            "trench (a long, double-breasted coat with a belt, often water-resistant, originally military style)",
            "puffer (a quilted, insulated jacket or coat, filled with down or synthetic fibers for warmth)",
            "windbreaker (a lightweight, wind-resistant jacket, usually with a zip front and elastic cuffs)"
        ],
        "closure": [
            "button (fastens with one or more rows of buttons down the front)",
            "zipper (fastens with a continuous interlocking metal or plastic teeth closure)",
            "belt (secures with a wrap-around belt and buckle at the waist)",
            "snap (fastens with press-stud snaps for quick open/close)",
            "toggle (secures with toggle fasteners—loops and horn or wooden pegs)"
        ],
        "length": [
            "hip",
            "thigh",
            "knee",
            "calf"
        ],
        "insulation": [
            "unlined (no extra layer inside, just the outer fabric)",
            "light (thin lining for mild warmth, not bulky)",
            "medium (moderate padding for cool weather, somewhat puffy)",
            "heavy (thick, bulky padding for very cold weather, very warm)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": list(CLIP_MATERIAL_DESCRIPTIONS.keys())
    },
    "Bottoms": {
        "fit": [
            "skinny (very tight fit, hugs the body closely from waist to ankle)",
            "slim (slightly tight, but not as close-fitting as skinny)",
            "regular (standard fit, not too tight and not too loose)",
            "relaxed (looser than regular, gives extra room for comfort)",
            "baggy (very loose, lots of extra space around the legs)"
        ],
        "style": [  # Changed from "type" to "style"
            "jeans (denim trousers with rivets and five-pocket styling)",
            "chinos (casual cotton twill trousers with a flat front)",
            "slacks (tailored dress trousers, often wool or blend)",
            "cargos (loose-fit pants with large side patch pockets)",
            "leggings (tight, stretch knit pants often worn as activewear)"
        ],
        "closure": [
            "button (fastens at the waist with one or more buttons)",
            "zipper (fastens at the waist or fly with a zipper)",
            "drawstring (adjusts fit with a cord or tie at the waistband)",
            "elastic (features a stretch waistband with no added fasteners)"
        ],
        "length": [
            "short",
            "capri",
            "ankle",
            "full"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": list(CLIP_MATERIAL_DESCRIPTIONS.keys())
    },
    "Skirts": {
        "fit": [
            "fitted",
            "regular",
            "flowy"
        ],
        "length": [
            "mini (hem ends well above the knee, typically mid-thigh)",
            "knee (hem ends at or just above the knee)",
            "midi (hem falls between the knee and the ankle, often mid-calf)",
            "maxi (hem extends to the ankle or floor)"
        ],
        "closure": [
            "pullover (designed to be pulled over the head or up the legs, no opening)",
            "zipper (fastens with a zipper)",
            "button (fastens with buttons)",
            "tie (fastens by tying fabric or strings)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": list(CLIP_MATERIAL_DESCRIPTIONS.keys())
    },
    "Dresses & Rompers": {
        "sleeve": [
            "sleeveless (no sleeves, exposing the arms)",
            "short (sleeves ending above the elbow)",
            "long (sleeves extending to the wrist)"
        ],
        "type": [  # Changed from "style" to "type"
            "shift (straight, column-like silhouette, loose from shoulders)",
            "bodycon (very form-fitting, hugs the body’s curves)",
            "a-line (fitted at the bodice, flares gently toward hem)",
            "wrap (front panels overlap and tie at the waist)",
            "shirt (resembles a long button-front shirt, straight cut)",
        ],
        "length": [
            "mini (ends significantly above the knee, mid-thigh or higher)",
            "knee-length (ends at or just around the knee)",
            "midi (ends between the knee and the ankle, typically mid-calf)",
            "maxi (extends to the ankle or floor)"
        ],
        "closure": [
            "pullover (pulled over the head, no opening)",
            "zipper (fastens with a zipper)",
            "button (fastens with buttons)",
            "tie (fastens by tying fabric or strings)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": list(CLIP_MATERIAL_DESCRIPTIONS.keys())
    },
    "Footwear": {
        "style": [  # Changed from "type" to "style"
            "sneakers (casual athletic shoes with rubber sole)",
            "boots (sturdy shoes covering the ankle or higher)",
            "sandals (open‑toed, strap‑based shoes)",
            "loafers (slip‑on shoes with low heel)",
            "pumps (women’s shoes with moderate to high heels)",
            "flats (low‑heeled or no‑heel shoes)"

        ],
        "closure": [
            "lace-up (fastens with laces that are tied)",
            "slip-on (designed to be slipped on, no fasteners)",
            "buckle (fastens with a buckle and strap)",
            "zipper (fastens with a zipper)",
            "hook-loop (fastens with Velcro or hook-and-loop)"
        ],
        "height": [
            "low-top (upper ends below the ankle)",
            "mid-top (upper covers the ankle)",
            "high-top (upper extends above the ankle)"
        ],
        "toe": [
            "round toe (rounded front shape, curves gently)",
            "pointed toe (tapers sharply to a point)",
            "square toe (flat, squared-off front)",
            "open toe (no front covering, toes exposed)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": ["leather", "suede", "canvas", "knit", "polyester", "plastic", "blend", "faux leather"]
    },
    "Headwear": {
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": ["cotton", "wool", "knit", "denim", "leather", "polyester", "blend", "silk", "plastic"]
    },
    "Bags": {
        "closure": [
            "zipper (fastens with a zipper)",
            "snap (fastens with snap buttons)",
            "magnetic (fastens with hidden magnets)",
            "drawstring (adjusts fit with a cord or tie)",
            "buckle (fastens with a buckle)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": ["leather", "canvas", "denim", "polyester", "faux leather", "silk", "plastic"]
    },
    "Neckwear": {
        "closure": [
            "none (no fasteners, a continuous loop)",
            "tie (fastens by tying fabric or strings)",
            "clasp (fastens with a clasp or hook)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": ["silk", "wool", "knit", "cotton", "linen", "blend", "gold", "silver", "plastic"]
    },
    "Earwear": {
        "closure": [
            "none (no fasteners, just slips on)",
            "piercing (inserted through a piercing)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": ["gold", "silver", "plastic", "blend"]
    },
    "Wristwear": {
        "closure": [
            "clasp (fastens with a clasp or hook)",
            "buckle (fastens with a buckle)",
            "none (a continuous loop, like a cuff or bangle)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": ["gold", "silver", "leather", "plastic", "knit", "blend"]
    },
    "Handwear": {
        "closure": [
            "none (no fasteners, just slips on)",
            "strap (fastens with an adjustable strap)",
            "button (fastens with a button)",
            "zipper (fastens with a zipper)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": ["wool", "knit", "leather", "faux leather", "polyester", "spandex", "blend"]
    },
    "Eyewear": {
        "closure": [
            "none (no fasteners, worn on the face)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": ["plastic", "gold", "silver"]
    },
    "Socks & Hosiery": {
        "closure": [
            "none (no fasteners, worn by pulling on)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": ["cotton", "polyester", "spandex", "knit", "blend", "wool"]
    },
    "Other Accessories": {
        "closure": [
            "buckle (for a belt)",
            "clasp (for a brooch)",
            "none (for a ring, pocket square, or umbrella)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": ["leather", "denim", "gold", "silver", "plastic", "silk", "wool", "cotton", "blend"]
    },
    "Underwear & Swimwear": {
        "type": [
            "top (an upper-body garment such as a bra, bralette, or bikini top)",
            "bottom (a lower-body garment such as panties, thongs, briefs, boxers, or bikini bottoms)",
            "fullwear (a single-piece garment that covers both the upper and lower body, like a one-piece swimsuit, bodysuit, or teddy)"
        ],
        "fit": [
            "supportive (a bra with a rigid structure, molding, or underwire to provide lift and shaping)",
            "padded (a bra with thick, soft inserts or foam that adds volume and creates a fuller appearance)",
            "unlined (a bra that has no padding, no foam inserts, and a thin material that follows the natural shape)",
            "brief (underwear that provides extensive, full coverage for the rear and hips)",
            "thong (underwear with a minimal, narrow strip of fabric at the back that leaves the rear exposed)"
        ],
        "coverage": [
            "full coverage (provides extensive and comprehensive coverage, leaving very little skin exposed, like a full brief or high-waisted bottom)",
            "medium coverage (offers moderate and balanced coverage, typically revealing some skin at the hips or cheeks, like a classic bikini or hipster bottom)",
            "minimal coverage (exposes a significant portion of the body, often with a low-rise fit or a narrow cut, like a thong or string bikini)"
        ],
        "closure": [
            "hook-and-eye (a bra closure with a series of small hooks and loops, typically found on the back)",
            "clasp (a secure fastener, often made of plastic or metal, that interlocks to close a garment)",
            "tie (a closure made by knotting fabric or strings together, like on a bikini top)",
            "pull-on (a garment that stretches and has no visible fasteners, made to be pulled over the body or head)"
        ],
        "pattern": list(CLIP_PATTERN_DESCRIPTIONS.keys()),
        "material": ["lace", "satin", "cotton", "nylon", "spandex", "mesh", "blend"],
        "color_group": list(COLOR_GROUPS.keys())
    }
}



CATEGORY_TO_GROUP = {cat: group for group, items in CATEGORY_LIST_BY_GROUP.items() for cat in items}


pattern_keyword_map = {
    "graphic": {"graphic", "logo", "artwork", "illustration", "slogan", "word", "font", "text",},
    "striped": {"striped", "stripes", "pinstripe"},
    "checked": {"checked", "checkered", "checkerboard", "squares", "grid", "rectangles"},
    "plaid": {"plaid", "tartan"},
    "floral": {"floral", "flowers", "botanical", "leaves"},
    "polka dots": {"polka dot", "polka dots", "dots", "spotted"},
    "geometric": {"geometric", "triangles", "squares", "hexagons"},
    "paisley": {"paisley"},
    "animal print": {"animal print", "leopard", "zebra", "snake", "cheetah"},
    "tie-dye": {"tie-dye", "tie dye", "dyed"},
    "camouflage": {"camouflage", "camo"},
    "ombre": {"ombre", "gradient", "faded"},
    "color-block": {"color-block", "color block", "block color"},
    "jacquard": {"jacquard"},
    "houndstooth": {"houndstooth"},
    "batik": {"batik"},
    "textured": {"textured", "texture", "embossed", "quilted", "cable knit"},
    "solid": {"solid", "plain", "no pattern", "single color"},
}



## MODELS

In [3]:
florence_model_id = "microsoft/Florence-2-base"
florence_model = AutoModelForCausalLM.from_pretrained(florence_model_id, trust_remote_code=True)
florence_processor = AutoProcessor.from_pretrained(florence_model_id, trust_remote_code=True)


device = "cuda:0" if torch.cuda.is_available() else "cpu"
florence_model.to(device).eval()



device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("patrickjohncyh/fashion-clip").to(device)
clip_processor = CLIPProcessor.from_pretrained("patrickjohncyh/fashion-clip")



## FUNCTIONS

In [4]:
# def correct_category_with_keywords(description, predicted_category, confidence, keyword_to_category, threshold=0.5):
#     desc_lower = description.lower().strip()
#     desc_clean = re.sub(r'[\W_]+', ' ', desc_lower).strip()

#     if re.search(r'\bbutton[- ]?down shirt\b', desc_lower):
#         return "Button-down shirt", 0.9

#     if confidence < threshold:
#         best_match = None
#         best_score = 0

#         for kw, cat in keyword_to_category.items():
#             score = fuzz.token_set_ratio(desc_clean, kw)

#             if score > best_score:
#                 best_score = score
#                 best_match = cat

#         fuzzy_threshold = 85
#         if best_match and best_score > fuzzy_threshold:
#             return best_match, best_score / 100.0

#     return predicted_category, confidence

def correct_category_with_description(florence_description: str, predicted_category: str, confidence: float, keyword_to_category: dict, confidence_threshold: float = 0.5):
    categories = list(CATEGORY_DESCRIPTIONS.keys())

    clip_text_features = clip_model.get_text_features(
        clip_processor.tokenizer(
            categories,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=77
        ).to(device)['input_ids']
    )

    description_features = clip_model.get_text_features(
        clip_processor.tokenizer(
            florence_description,
            return_tensors='pt',
            truncation=True,
            max_length=77
        ).to(device)['input_ids']
    )

    clip_text_features /= clip_text_features.norm(p=2, dim=-1, keepdim=True)
    description_features /= description_features.norm(p=2, dim=-1, keepdim=True)

    similarity_with_description = (description_features @ clip_text_features.T).squeeze(0)
    top_idx_desc = similarity_with_description.argmax().item()
    predicted_category_desc = categories[top_idx_desc]
    confidence_desc = similarity_with_description[top_idx_desc].item()

    desc_lower = florence_description.lower().strip()
    desc_clean = re.sub(r'[\W_]+', ' ', desc_lower).strip()

    first_sentence = desc_lower.split('.')[0].strip()

    if re.search(r'\bbutton[- ]?down shirt\b', first_sentence):
        return "Button-down shirt", 0.9

    if confidence < confidence_threshold:
        best_match = None
        best_score = 0

        for kw, cat in keyword_to_category.items():
            score = fuzz.token_set_ratio(first_sentence, kw)

            if score > best_score:
                best_score = score
                best_match = cat

        fuzzy_threshold = 85
        if best_match and best_score > fuzzy_threshold:
            return best_match, best_score / 100.0

    if confidence < confidence_threshold and confidence_desc > confidence:
        return predicted_category_desc, confidence_desc

    return predicted_category, confidence

def remove_transparency(img_path, bg_color=(255, 255, 255)):
    img = Image.open(img_path).convert("RGBA")
    bg = Image.new("RGBA", img.size, bg_color + (255,))
    bg.paste(img, mask=img.split()[-1])
    return bg.convert("RGB")

def extract_color_group(description: str, fg_image: Image.Image, n_colors=3):
    desc_lower = (description or "").lower()
    first_sentence = desc_lower.split('.', 1)[0]

    arr = np.array(fg_image)
    if arr.shape[2]==4:
        mask = arr[:,:,3]>0
        pixels = arr[mask][:,:3]
    else:
        pixels = arr.reshape(-1,3)
    if len(pixels)==0:
        return "unknown","none",0.0

    kmeans = KMeans(n_clusters=n_colors, random_state=0, n_init=10).fit(pixels)
    counts = np.bincount(kmeans.labels_)
    main_rgb = kmeans.cluster_centers_[counts.argmax()].astype(int)
    visual_conf = counts.max()/counts.sum()

    def closest_color(requested_rgb):
      min_dist = float('inf')
      closest_name = None

      for name in webcolors.names("css3"):
          r, g, b = webcolors.name_to_rgb(name)
          dist = (r - requested_rgb[0])**2 + (g - requested_rgb[1])**2 + (b - requested_rgb[2])**2
          if dist < min_dist:
              min_dist = dist
              closest_name = name

      return closest_name

    rgb_value = tuple(main_rgb)
    color_name = closest_color(rgb_value)

    r,g,b = main_rgb/255.0
    h,s,v = colorsys.rgb_to_hsv(r,g,b)
    if s>0.5 and v>0.5:     group="brights"
    elif v>0.85 and s<0.25: group="pastels"
    elif s<0.2:             group="neutrals"
    elif v<0.3 and s>0.2:   group="darks"
    else:                   group="neutrals"
    source="visual_kmeans"

    for kw in COLOR_GROUPS_MAP["metallics"]:
        if kw in first_sentence:
            group = "metallics"
            source = "text_override"
            break

    return group, source, round(visual_conf,4)

def extract_pattern(description, predicted_group, foreground_image,
                    pattern_keyword_map=None, clip_label=None,
    clip_confidence=0.0,threshold=0.3):
    attr_candidates = [
              f"{pattern} ({CLIP_PATTERN_DESCRIPTIONS[pattern]})"
              for pattern in ATTRIBUTES[predicted_group]["pattern"]
          ]


    clip_inputs = clip_processor(
                        text=attr_candidates,
                        images=foreground_image,
                        return_tensors="pt",
                        padding=True
                    ).to(device)

    with torch.no_grad():
                      clip_outputs = clip_model(**clip_inputs)
                      logits_per_image = clip_outputs.logits_per_image
                      probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    clip_label = attr_candidates[top_idx]
    clip_confidence = probs[top_idx].item()


    clip_pattern = clip_label.split(" (")[0]
    desc = (description or "").lower()
    first_sentence = desc.split('.', 1)[0]

    if "graphic" in first_sentence:
        return "graphic", 0.99


    if clip_label and clip_confidence >= threshold:

        return  clip_pattern, float(clip_confidence)


    for pat, kws in pattern_keyword_map.items():
         if pat in desc or pat.replace("-", " ") in desc:
             return pat, float(clip_confidence)
         for kw in kws:
             if kw in desc:
                 return pat, float(clip_confidence)


    scores = {pat: sum(1 for kw in kws if kw in desc)
              for pat, kws in pattern_keyword_map.items()}
    best, score = max(scores.items(), key=lambda x: x[1])
    if score >= 2:
        return best, float(clip_confidence)

    return "solid", float(clip_confidence)


def extract_material(predicted_group, canvas, description, clip_label=None, clip_confidence=0.0, predicted_category=None, threshold=0.5):
    attr_candidates = [
        f"{material} ({FULL_CLIP_MATERIAL_DESCRIPTIONS[material]})"
        for material in ATTRIBUTES[predicted_group]["material"]
    ]

    clip_inputs = clip_processor(
        text=attr_candidates,
        images=canvas,
        return_tensors="pt",
        padding=True
    ).to(device)
    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    clip_label = attr_candidates[top_idx]
    clip_confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs
    if clip_label and clip_confidence > threshold:
        return clip_label, clip_confidence

    if description:
        desc = description.lower()
        desc_clean = re.sub(r'[\W_]+', ' ', desc).strip()

        best_match = None
        best_score = 0

        for material, desc in FULL_CLIP_MATERIAL_DESCRIPTIONS.items():
            score = fuzz.ratio(desc_clean, desc.lower())
            if score > best_score:
                best_score = score
                best_match = material

        fallback_threshold = 70
        if best_score > fallback_threshold:
            return best_match, best_score / 100.0

    if predicted_category:
        default_mat = CATEGORY_DEFAULT_MATERIAL.get(predicted_category, "unknown")
        return default_mat, 0.5

    return clip_label, clip_confidence





def detect_sleeve_length(description,
                         attr_candidates, canvas,
                         predicted_label=None,
                         confidence=None,
                         predicted_category=None,
                         threshold=0.6):

    clip_inputs = clip_processor(
        text=attr_candidates,
        images=canvas,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    predicted_label = attr_candidates[top_idx]
    confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    desc = description.lower() if description else ""


    patterns = {
        "long": [
            r"\blong[- ]?sleeved\b",
            r"\bwith long sleeves\b",
            r"\bthe sleeves are long\b",
            r"\bsleeves are long\b",
        ],
        "short": [
            r"\bshort[- ]?sleeved\b",
            r"\bwith short sleeves\b",
            r"\bthe sleeves are short\b",
            r"\bcap[- ]?sleeved\b",
            r"\bhalf[- ]?sleeved\b",
            r"\b3/4[- ]?sleeved\b",
            r"\bthree[- ]?quarter[- ]?sleeved\b",
            r"\bshort sleeves\b",
        ],
        "sleeveless": [
            r"\bsleeveless\b",
            r"\bno[- ]?sleeves\b",
            r"\bstrapless\b",
        ],
    }

    for sleeve_type, regexes in patterns.items():
        for rx in regexes:
            if re.search(rx, desc):
                 return sleeve_type, "florence"




    if predicted_label is not None and confidence is not None and confidence >= threshold:
        label = predicted_label.lower()
        if "long" in label:
            return "long", "clip"
        if "short" in label:
            return "short", "clip"
        if "sleeveless" in label:
            return "sleeveless", "clip"

    return "sleeveless"




def detect_tops_closure(description,
                        attr_candidates, canvas,
                        predicted_label=None,
                        confidence=None,
                        predicted_category=None,
                        threshold=0.7):

    clip_inputs = clip_processor(
        text=attr_candidates,
        images=canvas,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    predicted_label = attr_candidates[top_idx]
    confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    desc = description.lower() if description else ""
    button_keywords = ["button", "buttons", "buttoned", "button-down"]
    zipper_keywords = ["zipper", "zippered", "zipped", "zip"]
    tie_keywords = ["tie", "tied", "tying", "string", "drawstring"]


    if "hood" in desc and "zipper" not in desc:
        return "pullover"


    if any(re.search(r'\b' + kw + r'\b', desc) for kw in button_keywords):
        return "button"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in zipper_keywords):
        return "zipper"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in tie_keywords):
        return "tie"


    if predicted_label is not None and confidence is not None and confidence >= threshold:
        label = predicted_label.lower()
        if "button" in label:
            return "button"
        if "zipper" in label or "zip" in label:
            return "zipper"
        if "tie" in label or "string" in label or "drawstring" in label:
            return "tie"
        if "pullover" in label:
            return "pullover"


    TOP_CATEGORIES = {
        "T-shirt", "Polo shirt", "Jersey",
        "Henley shirt", "Tank top", "Knit sweater", "Blouse",
        "Tunic", "Crop top", "Sleeveless top", "Pullover hoodie", "Turtleneck"
    }
    if predicted_category in TOP_CATEGORIES:
        return "pullover"


    return predicted_category



def detect_outerwear_closure(description, attr_candidates, canvas,
                             predicted_label=None, confidence=None, threshold=0.7):

    clip_inputs = clip_processor(
        text=attr_candidates,
        images=canvas,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    predicted_label = attr_candidates[top_idx]
    confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    desc = description.lower() if description else ""

    button_keywords = ["button", "buttons", "buttoned", "button-down"]
    zipper_keywords = ["zipper", "zippered", "zipped", "zip"]
    belt_keywords = ["belt", "belted", "buckle"]
    snap_keywords = ["snap", "snaps", "press-stud", "press stud"]
    toggle_keywords = ["toggle", "toggles"]




    if any(re.search(r'\b' + kw + r'\b', desc) for kw in button_keywords):
        return "button"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in zipper_keywords):
        return "zipper"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in belt_keywords):
        return "belt"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in snap_keywords):
        return "snap"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in toggle_keywords):
        return "toggle"


    if predicted_label and confidence is not None and confidence >= threshold:
        label = predicted_label.lower()
        if "button" in label:
            return "button"
        if "zipper" in label or "zip" in label:
            return "zipper"
        if "belt" in label or "buckle" in label:
            return "belt"
        if "snap" in label:
            return "snap"
        if "toggle" in label:
            return "toggle"


    return predicted_label



def detect_bottoms_closure(description, attr_candidates, canvas,
                           predicted_label=None, confidence=None, threshold=0.6):

    clip_inputs = clip_processor(
        text=attr_candidates,
        images=canvas,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    predicted_label = attr_candidates[top_idx]
    confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    closure_keywords = {
        "button": ["button", "buttons", "buttoned"],
        "zipper": ["zipper", "zippered", "zipped", "zip"],
        "drawstring": ["drawstring", "string", "tie", "tied"],
        "elastic": ["elastic", "waistband", "pull-on", "pull on"]
    }

    desc_lower = description.lower()

    threshold = 80

    for closure_type, keywords in closure_keywords.items():
        for keyword in keywords:
            ratio = fuzz.token_set_ratio(desc_lower, keyword)

            if ratio >= threshold:
                return closure_type


    if predicted_label and confidence is not None and confidence >= threshold:
        label = predicted_label.lower()
        if "button" in label:
            return "button"
        if "zipper" in label or "zip" in label:
            return "zipper"
        if "drawstring" in label or "string" in label or "tie" in label:
            return "drawstring"
        if "elastic" in label or "pull-on" in label or "pull on" in label:
            return "elastic"


    return predicted_label

def detect_skirt_closure(description, attr_candidates, canvas,
                         predicted_label=None, confidence=None, threshold=0.7):

    clip_inputs = clip_processor(
        text=attr_candidates,
        images=canvas,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    predicted_label = attr_candidates[top_idx]
    confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    desc = description.lower() if description else ""

    button_keywords = ["button", "buttons", "buttoned"]
    zipper_keywords = ["zipper", "zippered", "zipped", "zip"]
    tie_keywords = ["tie", "tied", "tying", "string", "drawstring"]
    pullover_keywords = ["pullover", "pull-on", "pull on", "elastic"]


    if any(re.search(r'\b' + kw + r'\b', desc) for kw in button_keywords):
        return "button"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in zipper_keywords):
        return "zipper"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in tie_keywords):
        return "tie"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in pullover_keywords):
        return "pullover"


    if predicted_label and confidence is not None and confidence >= threshold:
        label = predicted_label.lower()
        if "button" in label:
            return "button"
        if "zipper" in label or "zip" in label:
            return "zipper"
        if "tie" in label or "string" in label or "drawstring" in label:
            return "tie"
        if "pullover" in label or "pull-on" in label or "pull on" in label or "elastic" in label:
            return "pullover"


    return "pullover"


def detect_dress_closure(description, attr_candidates, canvas, predicted_label=None, confidence=None, threshold=0.7):

    clip_inputs = clip_processor(
        text=attr_candidates,
        images=canvas,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    predicted_label = attr_candidates[top_idx]
    confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    desc = description.lower() if description else ""

    button_keywords = ["button", "buttons", "buttoned"]
    zipper_keywords = ["zipper", "zippered", "zipped", "zip"]
    tie_keywords = ["tie", "tied", "tying", "string", "drawstring", "wrap"]
    pullover_keywords = ["pullover", "pull-on", "pull on", "elastic"]


    if any(re.search(r'\b' + kw + r'\b', desc) for kw in button_keywords):
        return "button"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in zipper_keywords):
        return "zipper"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in tie_keywords):
        return "tie"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in pullover_keywords):
        return "pullover"


    if predicted_label and confidence is not None and confidence >= threshold:
        label = predicted_label.lower()
        if "button" in label:
            return "button"
        if "zipper" in label or "zip" in label:
            return "zipper"
        if "tie" in label or "string" in label or "drawstring" in label or "wrap" in label:
            return "tie"
        if "pullover" in label or "pull-on" in label or "pull on" in label or "elastic" in label:
            return "pullover"


    return "pullover"





def detect_footwear_closure(description: str, attr_candidates, canvas,
                            predicted_label: str = None,
                            confidence: float = 0.0,
                            threshold: float = 0.7) -> str:

    clip_inputs = clip_processor(
        text=attr_candidates,
        images=canvas,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    predicted_label = attr_candidates[top_idx]
    confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    desc = description.lower() if description else ""


    lace_kw   = ["lace", "laces", "laced", "shoelace", "shoelaces", "tie", "tied"]
    buckle_kw = ["buckle", "buckled", "buckle closure"]
    strap_kw  = ["strap", "straps", "hook-and-loop", "velcro"]
    zip_kw    = ["zipper", "zippered", "zipped", "zip"]
    slip_kw   = ["slip-on", "slip on", "no laces", "no closure"]

    if any(re.search(rf"\b{kw}\b", desc) for kw in lace_kw):
        return "lace-up"
    if any(re.search(rf"\b{kw}\b", desc) for kw in buckle_kw):
        return "buckle"
    if any(re.search(rf"\b{kw}\b", desc) for kw in strap_kw):
        return "hook-loop"
    if any(re.search(rf"\b{kw}\b", desc) for kw in zip_kw):
        return "zipper"
    if any(re.search(rf"\b{kw}\b", desc) for kw in slip_kw):
        return "slip-on"


    if predicted_label and confidence >= threshold:
        lbl = predicted_label.lower()
        if "lace" in lbl:
            return "lace-up"
        if "slip" in lbl:
            return "slip-on"
        if "buckle" in lbl:
            return "buckle"
        if "zip" in lbl:
            return "zipper"
        if "hook" in lbl or "loop" in lbl or "velcro" in lbl:
            return "hook-loop"


    if "oxford" in desc or "dress shoe" in desc:
        return "lace-up"
    if "loafer" in desc or "moccasin" in desc:
        return "slip-on"
    if "boot" in desc and "zipper" not in desc:
        return "lace-up"
    if "sandal" in desc:
        return "buckle" if "buckle" in desc else "hook-loop"


    return predicted_label

def detect_accessory_closure(description, attr_candidates, canvas, predicted_group, predicted_label=None, confidence=None, threshold=0.7):
    clip_inputs = clip_processor(
        text=attr_candidates,
        images=canvas,
        return_tensors="pt",
        padding=True,
    ).to(device)
    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]
    top_idx = probs.argmax().item()
    predicted_label = attr_candidates[top_idx]
    confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    if confidence >= threshold:
        return predicted_label
    closure_candidates = ATTRIBUTES.get(predicted_group, {}).get("closure", []) if ATTRIBUTES and predicted_group else []
    if confidence is not None and confidence >= threshold:
        label = predicted_label.lower()
        for full_closure_string in closure_candidates:
            main_keyword = full_closure_string.split(" ")[0].lower()
            if main_keyword in label or ("hook-and-eye" in main_keyword and "hook-and-eye" in label) or ("pull-on" in main_keyword and "pull-on" in label):
                return main_keyword

    desc = description.lower() if description else ""
    desc_clean = re.sub(r'[\W_]+', ' ', desc)
    closure_map = {
        "zipper": ["zipper", "zipped", "zip", "zip-up", "front-zip", "back-zip", "zip front", "zip closure", "zip fastener"],
        "tie": ["tie", "tied", "tying", "drawstring", "self-tie", "ribbon tie", "tie-up", "string closure", "bow tie"],
        "button": ["button", "buttons", "buttoned", "button-front", "button-down", "snap button", "front-button closure", "button fly", "cuff buttons", "shirt buttons"],
        "pullover": ["pullover", "pull over", "overhead", "slip-over", "pull-over style", "no fasteners", "no closure"],
        "belt": ["belt", "belted", "tie waist", "sash", "waist tie", "adjustable belt"],
        "snap": ["snap", "snaps", "snap fastener", "snap closure", "press stud", "popper"],
        "toggle": ["toggle", "toggles", "toggle closure", "duffle", "loop-and-toggle"],
        "elastic": ["elastic", "stretch waistband", "stretchy band", "stretch band", "elasticated", "elastic closure", "smocked waistband"],
        "lace-up": ["lace-up", "lace up", "laces", "laced", "shoe laces", "corset lacing", "criss-cross laces", "ghillie laces", "speed laces"],
        "slip-on": ["slip-on", "slip on", "no fastener", "no fasteners", "no closure", "slips-on", "unfastened", "no laces", "easy to wear"],
        "buckle": ["buckle", "buckled", "buckle strap", "buckle closure", "adjustable buckle", "metal buckle"],
        "hook-loop": ["hook-loop", "hook and loop", "velcro", "velcro strap", "hook-and-loop fastener"],
        "none": ["none", "no adjustable closure", "slips on", "worn by pulling on", "no fastening", "open front"],
        "adjustable strap": ["adjustable strap", "strap", "adjustable straps", "straps", "adjustable shoulder strap", "shoulder straps", "removable straps"],
        "clasp": ["clasp", "hook", "clasp fastener", "metal clasp", "simple clasp", "lobster clasp", "toggle clasp", "push-button clasp"],
        "piercing": ["piercing", "pierced", "for pierced ears", "pierce", "ear piercing"],
        "magnetic": ["magnetic", "magnetic closure", "magnetic snap"],
        "hook-and-eye": ["hook-and-eye", "hook and eye", "hook and clasp", "bra clasp", "garter hook", "garter clasp"],
        "pull-on": ["pull-on", "pull on", "worn by pulling on", "stretch fit", "easy on and off", "no zippers or buttons"]
    }

    FUZZY_THRESHOLD = int(threshold * 100)
    best_match = ("unknown", 0)
    for full_closure_string in closure_candidates:
        main_keyword = full_closure_string.split(" ")[0].lower()
        if main_keyword in closure_map:
            keywords = closure_map[main_keyword]
        else:
            keywords = [main_keyword]
        for kw in keywords:
            score = fuzz.partial_ratio(kw, desc_clean)
            if score > best_match[1]:
                best_match = (main_keyword, score)

    if best_match[1] >= FUZZY_THRESHOLD:
        return best_match[0]

    if predicted_group in ["Earwear", "Eyewear", "Socks & Hosiery"]:
        return "none"

    return predicted_group

def detect_tops_length(image, attr_candidates, description, clip_label=None, clip_confidence=0.0, threshold=0.7):

    clip_inputs = clip_processor(
         text=attr_candidates,
         images=image,
         return_tensors="pt",
         padding=True
     ).to(device)

    with torch.no_grad():
         clip_outputs = clip_model(**clip_inputs)
         logits_per_image = clip_outputs.logits_per_image
         probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    clip_label = attr_candidates[top_idx]
    clip_confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    if clip_label and clip_confidence > threshold:
        return clip_label.split('(')[0].strip().lower(), "clip"

    try:
        width, height = image.size
        aspect_ratio = height / width
        if aspect_ratio < 0.9:
            return "crop", "visual"
        elif aspect_ratio < 1.2:
            return "standard", "visual"
        else:
            return "tunic", "visual"
    except:
        pass

    desc = description.lower()
    length_keywords = {
        "crop": ["crop", "cropped", "above waist", "short top"],
        "standard": ["standard", "hip", "at waist", "regular"],
        "tunic": ["tunic", "long top", "mid-thigh", "past hips"]
    }
    for length, keywords in length_keywords.items():
        if any(kw in desc for kw in keywords):
            return length, "keyword"



    return "standard", "fallback"


def detect_outerwear_length(image, attr_candidates, description, clip_label=None, clip_confidence=0.0, threshold=0.5):

    clip_inputs = clip_processor(
         text=attr_candidates,
         images=image,
         return_tensors="pt",
         padding=True
     ).to(device)
    with torch.no_grad():
         clip_outputs = clip_model(**clip_inputs)
         logits_per_image = clip_outputs.logits_per_image
         probs = logits_per_image.softmax(dim=1)[0]
    top_idx = probs.argmax().item()
    clip_label = attr_candidates[top_idx]
    clip_confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    try:

        arr = np.array(image.convert("L"))
        mask = arr < 250
        ys, xs = np.where(mask)
        if xs.size and ys.size:
            h = ys.max() - ys.min() + 1
            w = xs.max() - xs.min() + 1
            ar = h / w

            if ar < 1.0:
                return "hip",   "visual", 0.6
            elif ar < 1.5:
                return "thigh", "visual", 0.6
            elif ar < 2.0:
                return "knee",  "visual", 0.6
            else:
                return "calf",  "visual", 0.6
    except Exception:
        pass

    desc = description.lower()
    length_keywords = {
        "hip": ["hip length", "hip level", "covers hips"],
        "thigh": ["thigh length", "upper thigh", "mid-thigh"],
        "knee": ["knee length", "at knee", "knee-high", "just below knee"],
        "calf": ["calf length", "mid-calf", "below calf",]
    }
    for length, keywords in length_keywords.items():
        if any(kw in desc for kw in keywords):
            return length, "keyword", clip_confidence




    if clip_label and clip_confidence > threshold:
        return clip_label.split('(')[0].strip().lower(), "clip",clip_confidence


    return "hip", "fallback" , clip_confidence


def detect_bottoms_length(image, description, category, attr_candidates,
                          clip_label=None, clip_confidence=0.0, threshold=0.9):

    clip_inputs = clip_processor(
     text=attr_candidates,
     images=image,
     return_tensors="pt",
     padding=True
 ).to(device)
    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    clip_label = attr_candidates[top_idx]
    clip_confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    if clip_label and clip_confidence > threshold:
        return clip_label.split('(')[0].strip().lower(), "clip",clip_confidence



    try:

        arr = np.array(image.convert("L"))
        mask = arr < 250
        ys, xs = np.where(mask)
        if xs.size and ys.size:
            h = ys.max() - ys.min() + 1
            w = xs.max() - xs.min() + 1
            ar = h / w

            if ar < 0.8:
                return "short",  "visual", 0.6

            elif ar < 1.0:
                return "capri", "visual", 0.6

            elif ar < 1.3:
                return "ankle", "visual", 0.6

            else:
                return "full",  "visual", 0.6
    except Exception:
        pass

    # Geometry failed (no foreground pixels, or an exception). Fall back to
    # the CLIP prediction even though it scored below threshold: returning
    # None here crashes the caller, which unpacks three values.
    if clip_label:
        return clip_label.split('(')[0].strip().lower(), "fallback", clip_confidence
    return "full", "fallback", 0.0

def detect_skirt_length(image, attr_candidates, description, clip_label=None, clip_confidence=0.0, threshold=0.9):

    clip_inputs = clip_processor(
      text=attr_candidates,
      images=image,
      return_tensors="pt",
      padding=True
  ).to(device)
    with torch.no_grad():
            clip_outputs = clip_model(**clip_inputs)
            logits_per_image = clip_outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1)[0]
    top_idx = probs.argmax().item()
    clip_label = attr_candidates[top_idx]
    clip_confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs


    desc = description.lower()
    length_keywords = {
        "mini": ["mini", "above knee", "mid-thigh"],
        "knee": ["knee", "at knee", "just above knee"],
        "midi": ["midi", "mid-calf", "below knee"],
        "maxi": ["maxi", "ankle", "floor length","long skirt "]
    }
    for length, keywords in length_keywords.items():
        if any(kw in desc for kw in keywords):
            return length, "keyword",1.0

    try:
        width, height = image.size
        aspect_ratio = height / width
        if aspect_ratio < 1.0:
            return "mini", "visual",0.6
        elif aspect_ratio < 1.5:
            return "knee", "visual",0.6
        elif aspect_ratio < 2.0:
            return "midi", "visual",0.6
        else:
            return "maxi", "visual",0.6
    except:
        pass



    if clip_label and clip_confidence > threshold:
        return clip_label.split('(')[0].strip().lower(), "clip",clip_confidence

    return "knee", "fallback",0.5


def detect_dress_length(image, attr_candidates, description, category, clip_label=None, clip_confidence=0.0, threshold=0.9):


    clip_inputs = clip_processor(
        text=attr_candidates,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    clip_label = attr_candidates[top_idx]
    clip_confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    desc = description.lower()

    length_keywords = {
        "maxi":        [r"maxi", r"floor length", r"ankle length", r"full length"],
        "midi":        [r"midi", r"mid-calf", r"tea length", r"below the knees", r"below knee"],
        "knee-length": [r"knee length", r"at knee", r"knee-high", r"just below knee", r"knee-length"],
        "mini":        [r"mini", r"above knee", r"mid-thigh"]
    }
    for length, patterns in length_keywords.items():
        for pat in patterns:
            if re.search(rf"\b{pat}\b", desc):
                if length == "knee-length" and (re.search(r"below the knees", desc) or re.search(r"midi", desc) or re.search(r"mid-calf", desc)):
                    continue
                return length, "keyword",1.0


    if clip_label and clip_confidence > threshold:
        return clip_label.split('(')[0].strip().lower(), "clip",clip_confidence


    try:
        width, height = image.size
        aspect_ratio = height / width
        if category == "Romper":
            return ("short", "visual") if aspect_ratio < 1.2 else ("standard", "visual")
        if aspect_ratio < 1.0:
            return "mini", "visual",0.6
        elif aspect_ratio < 1.5:
            return "knee-length", "visual",0.6
        elif aspect_ratio < 2.0:
            return "midi", "visual",0.6
        else:
            return "maxi", "visual",0.6
    except:
        pass

    return "knee-length", "fallback",0.5

def detect_tops_fit(description, attr_candidates, image, clip_label = None, confidence =0.0 , threshold=0.7):

    clip_inputs = clip_processor(
        text=attr_candidates,
        images= image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    clip_label = attr_candidates[top_idx]
    confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    desc = description.lower() if description else ""


    slim_keywords = ["slim", "fitted", "close fit", "tailored"]
    regular_keywords = ["regular", "standard fit", "classic fit"]
    relaxed_keywords = ["relaxed", "loose", "easy fit"]
    oversized_keywords = ["oversized", "very loose", "baggy", "hangs away"]


    if any(re.search(r'\b' + kw + r'\b', desc) for kw in slim_keywords):
        return "slim"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in regular_keywords):
        return "regular"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in relaxed_keywords):
        return "relaxed"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in oversized_keywords):
        return "oversized"


    if clip_label and confidence is not None and confidence >= threshold:
        label = clip_label.lower()
        if "slim" in label or "fitted" in label:
            return "slim"
        if "regular" in label or "standard" in label or "classic" in label:
            return "regular"
        if "relaxed" in label or "loose" in label or "easy" in label:
            return "relaxed"
        if "oversized" in label or "very loose" in label or "baggy" in label or "hangs away" in label:
            return "oversized"


    return "regular"


def detect_bottoms_fit(description, attr_candidates, image,clip_label = None,confidence =0.0 , threshold=0.7):

    clip_inputs = clip_processor(
        text=attr_candidates,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    clip_label = attr_candidates[top_idx]
    confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    desc = description.lower() if description else ""

    skinny_keywords = ["skinny", "super skinny", "tightest"]
    slim_keywords = ["slim", "fitted", "close fit", "tailored"]
    regular_keywords = ["regular", "standard fit", "classic fit"]
    relaxed_keywords = ["relaxed", "loose", "easy fit"]
    baggy_keywords = ["baggy", "very loose", "extra loose", "oversized"]


    if any(re.search(r'\b' + kw + r'\b', desc) for kw in skinny_keywords):
        return "skinny"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in slim_keywords):
        return "slim"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in regular_keywords):
        return "regular"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in relaxed_keywords):
        return "relaxed"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in baggy_keywords):
        return "baggy"


    if clip_label and confidence is not None and confidence >= threshold:
        label = clip_label.lower()
        if "skinny" in label:
            return "skinny"
        if "slim" in label or "fitted" in label:
            return "slim"
        if "regular" in label or "standard" in label or "classic" in label:
            return "regular"
        if "relaxed" in label or "loose" in label or "easy" in label:
            return "relaxed"
        if "baggy" in label or "very loose" in label or "extra loose" in label or "oversized" in label:
            return "baggy"


    return "regular"


def detect_skirt_fit(description, attr_candidates, image, clip_label = None,confidence =0.0 , threshold=0.7):

    clip_inputs = clip_processor(
        text=attr_candidates,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    clip_label = attr_candidates[top_idx]
    confidence = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs

    desc = description.lower() if description else ""

    fitted_keywords = ["fitted", "tight", "bodycon", "pencil"]
    regular_keywords = ["regular", "standard fit", "classic fit"]
    flowy_keywords = ["flowy", "loose", "draped", "full", "a-line", "pleated"]


    if any(re.search(r'\b' + kw + r'\b', desc) for kw in fitted_keywords):
        return "fitted"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in regular_keywords):
        return "regular"
    if any(re.search(r'\b' + kw + r'\b', desc) for kw in flowy_keywords):
        return "flowy"


    if clip_label and confidence is not None and confidence >= threshold:
        label = clip_label.lower()
        if "fitted" in label or "tight" in label or "bodycon" in label or "pencil" in label:
            return "fitted"
        if "regular" in label or "standard" in label or "classic" in label:
            return "regular"
        if "flowy" in label or "loose" in label or "draped" in label or "full" in label or "a-line" in label or "pleated" in label:
            return "flowy"


    return "regular"


def detect_underwear_fit(description, attr_candidates, image, clip_label=None, confidence=0.0, threshold=0.7):
    if clip_label is None:
        clip_inputs = clip_processor(
            text=attr_candidates,
            images=image,
            return_tensors="pt",
            padding=True
        ).to(device)

        with torch.no_grad():
            clip_outputs = clip_model(**clip_inputs)
            logits_per_image = clip_outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1)[0]

        top_idx = probs.argmax().item()
        clip_label = attr_candidates[top_idx]
        confidence = probs[top_idx].item()
        del clip_outputs, clip_inputs, probs

    if confidence >= threshold:
        label_lower = clip_label.lower()
        if label_lower in [c.lower() for c in attr_candidates]:
            return label_lower
    desc = description.lower() if description else ""
    desc_clean = re.sub(r'[\W_]+', ' ', desc)

    fit_keywords = {
        "supportive": ["supportive", "underwire", "shaping", "firm", "high-impact", "structured"],
        "padded": ["padded", "push-up", "molded cup", "add volume", "cushioning", "foam inserts"],
        "unlined": ["unlined", "no padding", "wireless", "natural shape", "soft cup"],
        "brief": ["brief", "full coverage", "high-waisted", "bikini brief", "classic brief"],
        "thong": ["thong", "g-string", "t-back", "tanga", "minimal coverage", "v-string"]
    }
    best_match = ("unknown", 0)

    for fit_type, keywords in fit_keywords.items():
        for keyword in keywords:
            score = fuzz.partial_ratio(keyword, desc_clean)
            if score > best_match[1]:
                best_match = (fit_type, score)

    if best_match[1] >= threshold*100:
        return best_match[0]
    else:
        return "supportive"

def detect_item_type(predicted_group, description, attr_candidates, image, clip_label=None, confidence=0.0, threshold=0.7):

    type_candidates = ATTRIBUTES.get(predicted_group, {}).get("type", [])

    if not type_candidates:
        return "N/A"

    if clip_label is None:
        clip_inputs = clip_processor(
            text=attr_candidates,
            images=image,
            return_tensors="pt",
            padding=True
        ).to(device)

        with torch.no_grad():
            clip_outputs = clip_model(**clip_inputs)
            logits_per_image = clip_outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1)[0]

        top_idx = probs.argmax().item()
        clip_label = attr_candidates[top_idx]
        confidence = probs[top_idx].item()
        del clip_outputs, clip_inputs, probs

    if confidence >= threshold:
        label_match = next((candidate for candidate in type_candidates if candidate.lower().split(" ")[0] in clip_label.lower()), None)
        if label_match:
            return label_match.split(" ")[0].lower()

    desc = description.lower() if description else ""
    desc_clean = re.sub(r'[\W_]+', ' ', desc)

    type_keywords = {}
    for item in type_candidates:
        main_keyword = item.lower()
        if main_keyword == "shift":
            type_keywords[main_keyword] = ["shift", "column", "straight dress", "sack dress"]
        elif main_keyword == "bodycon":
            type_keywords[main_keyword] = ["bodycon", "form-fitting", "form fitting", "tight dress", "figure-hugging"]
        elif main_keyword == "a-line":
            type_keywords[main_keyword] = ["a-line", "a line", "flares", "fit and flare"]
        elif main_keyword == "wrap":
            type_keywords[main_keyword] = ["wrap", "wrap dress", "wrap-around"]
        elif main_keyword == "shirt":
            type_keywords[main_keyword] = ["shirt", "shirt dress", "button-front", "collared dress"]
        elif main_keyword == "top":
            type_keywords[main_keyword] = ["top", "t-shirt", "tshirt", "bralette", "bikini top", "tank top", "blouse", "sweater", "hoodie"]
        elif main_keyword == "bottom":
            type_keywords[main_keyword] = ["bottom", "brief", "thong", "bikini bottom", "shorts", "skirt", "pants", "trousers", "jeans", "leggings"]
        elif main_keyword == "fullwear":
            type_keywords[main_keyword] = ["fullwear", "one piece", "one-piece", "bodysuit", "jumpsuit", "romper", "playsuit", "unitard"]

    best_match = ("unknown", 0)

    for type_name, keywords in type_keywords.items():
        for keyword in keywords:
            score = fuzz.partial_ratio(keyword, desc_clean)
            if score > best_match[1]:
                best_match = (type_name, score)

    if best_match[1] >= threshold*100:
        return best_match[0]
    else:
        return "unknown"

def detect_item_style(predicted_group, description, attr_candidates, image, clip_label=None, confidence=0.0, threshold=0.7):
    style_candidates = ATTRIBUTES.get(predicted_group, {}).get("style", [])
    if not style_candidates:
        style_candidates = ATTRIBUTES.get(predicted_group, {}).get("type", [])
        if not style_candidates:
            return "N/A"

    if clip_label is None:
        clip_inputs = clip_processor(
            text=attr_candidates,
            images=image,
            return_tensors="pt",
            padding=True
        ).to(device)

        with torch.no_grad():
            clip_outputs = clip_model(**clip_inputs)
            logits_per_image = clip_outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1)[0]

        top_idx = probs.argmax().item()
        clip_label = attr_candidates[top_idx]
        confidence = probs[top_idx].item()
        del clip_outputs, clip_inputs, probs

    if confidence >= threshold:
        label_match = next((candidate for candidate in style_candidates if candidate.lower().split(" ")[0] in clip_label.lower()), None)
        if label_match:
            return label_match.split(" ")[0].lower()

    desc = description.lower() if description else ""
    desc_clean = re.sub(r'[\W_]+', ' ', desc)

    style_keywords = {
        "blazer": ["blazer", "tailored jacket", "sport coat", "suit jacket", "dress jacket"],
        "bomber": ["bomber", "bomber jacket", "waist-length jacket", "flight jacket", "varsity jacket"],
        "parka": ["parka", "insulated coat", "anorak", "heavy coat", "winter coat", "down coat"],
        "trench": ["trench", "trench coat", "double-breasted coat", "raincoat", "long coat", "belted coat"],
        "puffer": ["puffer", "puffer jacket", "quilted jacket", "down jacket", "padded jacket"],
        "windbreaker": ["windbreaker", "windbreaker jacket", "lightweight jacket", "rain jacket", "shell jacket"],

        "jeans": ["jeans", "denim trousers", "denim pants", "denim jeans", "skinnies", "straight-leg", "bootcut"],
        "chinos": ["chinos", "cotton twill trousers", "chino pants", "khaki pants", "dress pants"],
        "slacks": ["slacks", "tailored dress trousers", "dress pants", "formal trousers", "business pants"],
        "cargos": ["cargos", "cargo pants", "loose-fit pants", "utility pants", "pocket pants"],
        "leggings": ["leggings", "stretch knit pants", "yoga pants", "tights", "jeggings"],

        "shift": ["shift", "shift dress", "column-like", "straight dress", "simple dress", "sack dress"],
        "bodycon": ["bodycon", "bodycon dress", "form-fitting", "tight dress", "figure-hugging"],
        "a-line": ["a-line", "a line", "a-line dress", "flares", "fit and flare"],
        "wrap": ["wrap", "wrap dress", "wrap-around"],
        "shirt": ["shirt", "shirt dress", "button-front", "collared dress", "tunic dress"],
        "maxi": ["maxi", "maxi dress", "floor-grazing", "long dress", "full-length gown"],

        "sneakers": ["sneakers", "athletic shoes", "running shoes", "trainers", "tennis shoes", "gym shoes"],
        "boots": ["boots", "sturdy shoes", "ankle boots", "combat boots", "chelsea boots", "knee-high boots"],
        "sandals": ["sandals", "open-toed", "strap-based", "flips flops", "slides", "espadrilles"],
        "loafers": ["loafers", "slip-on shoes", "moccasins", "driving shoes", "penny loafers"],
        "pumps": ["pumps", "high heels", "stiletto", "stilettos", "court shoes"],
        "flats": ["flats", "low-heeled shoes", "ballet flats", "skimmers", "dolly shoes", "mules"]
    }
    best_match = ("unknown", 0)
    for style_type, keywords in style_keywords.items():
        for keyword in keywords:
            score = fuzz.partial_ratio(keyword, desc_clean)
            if score > best_match[1]:
                best_match = (style_type, score)

    if best_match[1] >= threshold*100:
        return best_match[0]
    else:
        return "unknown"


def detect_item_coverage(predicted_group, description, attr_candidates, image, clip_label=None, confidence=0.0, threshold=0.7):
    coverage_candidates = ATTRIBUTES.get(predicted_group, {}).get("coverage", [])
    if not coverage_candidates:
        return "N/A"
    if clip_label is None:
        clip_inputs = clip_processor(
            text=attr_candidates,
            images=image,
            return_tensors="pt",
            padding=True
        ).to(device)

        with torch.no_grad():
            clip_outputs = clip_model(**clip_inputs)
            logits_per_image = clip_outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1)[0]

        top_idx = probs.argmax().item()
        clip_label = attr_candidates[top_idx]
        confidence = probs[top_idx].item()
        del clip_outputs, clip_inputs, probs

    if confidence >= threshold:
        label_match = next((candidate for candidate in coverage_candidates if candidate.lower().split(" ")[0] in clip_label.lower()), None)
        if label_match:
            return label_match.split(" ")[0].lower()

    desc = description.lower() if description else ""
    desc_clean = re.sub(r'[\W_]+', ' ', desc)

    coverage_keywords = {
        "full": [
            "full", "extensive coverage", "full coverage", "maximum coverage",
            "high coverage", "high rise", "full cut", "modest", "opaque",
            "longline", "full length", "over-the-knee", "ankles", "full-back"
        ],
        "medium": [
            "medium", "moderate coverage", "moderate", "standard coverage",
            "mid-rise", "regular cut", "cheeky", "half-back", "hip-hugger",
            "classic fit", "three-quarter", "knee-length"
        ],
        "minimal": [
            "minimal", "exposes a significant portion", "minimal coverage",
            "low coverage", "low rise", "g-string", "thong", "cheeky",
            "skimpy", "scanty", "micro", "briefs", "brazilian cut", "mini",
            "short length", "above-the-knee", "crop top"
        ]
    }
    best_match = ("unknown", 0)
    for coverage_type, keywords in coverage_keywords.items():
        for keyword in keywords:
            score = fuzz.partial_ratio(keyword, desc_clean)
            if score > best_match[1]:
                best_match = (coverage_type, score)

    if best_match[1] >= threshold*100:
        return best_match[0]
    else:
        return "medium"

## CHECKER

In [5]:
category_map_fuzz = [
    "T-shirt", "Polo shirt", "Jersey", "Button-down shirt", "Henley shirt",
    "Tank top", "Knit sweater", "Blouse", "Tunic", "Crop top",
    "Sleeveless top", "Pullover hoodie", "Turtleneck", "button-up shirt",
    "Dashiki tunic", "Ao dai tunic", "Huipil blouse", "Kente cloth top",
    "Denim jacket", "Leather jacket", "Puffer jacket", "Trench coat",
    "Peacoat", "Blazer", "Windbreaker jacket", "Cardigan sweater",
    "Vest", "Raincoat", "Parka", "Zippered hoodie", "Tracksuit jacket",
    "Coat", "Jacket", "Leather bomber jacket", "Leather coat", "Faux Fur Coat",
    "Kimono robe", "Sherwani coat",
    "Straight-leg jeans", "Skinny jeans", "Bootcut jeans", "Cargo pants",
    "Chino pants", "Dress pants", "Shorts", "Capri pants", "Leggings",
    "Joggers", "Denim shorts", "Sweatpants", "Trousers",
    "Lederhosen pants",
    "A-line skirt", "Pencil skirt", "Maxi skirt", "Mini skirt",
    "Pleated skirt", "Wrap skirt", "Denim skirt",
    "Kilt skirt", "Sarong wrap",
    "Strap dress", "Wrap dress", "T-shirt dress", "Maxi dress",
    "Midi dress", "Mini dress", "Cocktail dress", "Evening gown",
    "Sundress", "Shirt dress", "Sweater dress", "Overalls",
    "Jumpsuit", "Romper", "Denim dress",
    "Sari garment", "Hanbok dress", "Dirndl dress", "Cheongsam dress",
    "Boubou robe", "Caftan robe", "Abaya robe",
    "Sneakers", "Oxford dress shoes", "Ankle boots", "Knee-high boots",
    "Heel pumps", "Sandals", "Rain boots", "Loafers", "Ballet flats",
    "Wedges", "Espadrilles", "Slippers", "Suede dress shoes", "Dress shoe",
    "Baseball cap", "Beanie hat", "Bucket hat", "Sun hat", "Headband",
    "Headscarf", "Hijab head covering", "Beret", "Fedora", "Fez hat",
    "Turban", "Sombrero",
    "Handbag", "Backpack", "Tote bag", "Clutch bag", "Shoulder bag",
    "Crossbody bag", "Wallet",
    "Neck tie", "Bow tie", "Scarf", "Shawl", "Bandana", "Necklace",
    "Earrings", "Over-ear headphones", "Earbuds",
    "Wrist watch", "Bracelet",
    "Gloves", "Mittens",
    "Sunglasses", "Eyeglasses",
    "Knee-high socks", "Ankle socks", "Crew socks", "No-show socks",
    "Dress socks", "Stockings", "Tights",
    "Belt", "Ring", "Brooch", "Pocket square", "Umbrella",
    "Bra", "Panty", "One-piece swimsuit", "Boxers"
]

def is_item_accepted_with_description(
    image_without_background: Image.Image,
    florence_description: str,
    confidence_threshold: float = 0.7
) -> bool:
    # broad_categories = [
    #     "Apparel & Clothing: A wide range of garments and wearable textiles for the body, including tops (shirts, t-shirts, blouses, tank tops, sweaters, cardigans, jackets, coats, blazers), bottoms (pants, jeans, trousers, shorts, leggings, skirts), dresses, jumpsuits, and suits. Excludes unrelated objects, packaging, or fabric scraps.",
    #     "Footwear: Items worn on the feet for protection and style, such as athletic shoes, sneakers, running shoes, boots, dress shoes, loafers, heels, pumps, sandals, flip-flops, slippers, and flats.",
    #     "Bags & Luggage: Objects used for carrying personal belongings, including handbags, clutches, purses, backpacks, satchels, tote bags, duffel bags, suitcases, travel bags, briefcases, and wallets.",
    #     "Fashion Accessories: Items used to complement or enhance an outfit, often worn on the head, face, neck, or hands. This includes all forms of eyewear, such as sunglasses, shades, spectacles, aviator glasses, wayfarers, cat-eye glasses, and protective eye-wear. Other accessories include hats, caps, scarves, gloves, belts, neckties, bowties, jewelry (necklaces, earrings, bracelets, rings), and watches.",
    #     "Personal & Everyday Carry (EDC) Items: Small, portable items carried for utility or personal use. Examples include earbuds, headphones, keychains, portable chargers, pens, lighters, hand sanitizers, and compact utility tools. Excludes large electronics, furniture, and wearable accessories like watches.",
    #     "Household Goods: Items found in a home, including furniture (chairs, tables, sofas, beds), kitchenware (pots, pans, dishes, utensils), home decor (lamps, vases, picture frames), bedding, towels, and home appliances (refrigerators, microwaves, washing machines). Excludes all wearable fashion items.",
    #     "Electronics & Gadgets: Powered electronic devices, such as smartphones, tablets, laptops, desktop computers, television sets, video game consoles, digital cameras, drones, speakers, and smart home devices. Excludes smaller, non-powered personal items like wallets or keychains.",
    #     "Other: Items that do not fit into the primary categories, representing a background class for irrelevant or unclassifiable objects. This includes trash, packaging materials (boxes, plastic wrap), random fabric scraps, raw materials, or objects that are part of the background environment rather than the subject of the image."
    # ]

    broad_categories = [
        "Apparel & Clothing: Images of garments and fabrics worn on the body. This includes a variety of tops like t-shirts, blouses, jackets, and coats; bottoms like pants, jeans, skirts, and shorts; and full outfits such as dresses and suits.",
        "Footwear: Images of shoes being worn or displayed. This category includes all types of footwear such as athletic sneakers, running shoes, formal dress shoes, casual boots, sandals, and slippers.",
        "Bags & Luggage: Images of items used to carry belongings. This includes handbags, backpacks, clutches, suitcases for travel, duffel bags, tote bags, and wallets. The bags can be shown open, closed, held, or resting on a surface.",
        "Eyewear: Images of items worn on or covering the eyes. This is a dedicated category for all types of glasses, including **sleek black sunglasses with gold logos, classic aviator shades, polarized lenses, and prescription glasses**. These are often shown on a face, in a hand, or resting on a table.",
        "Headwear: Images of items worn on the head. This category specifically includes hats, caps, helmets, and beanies. Examples are straw hats with woven textures, fedoras, baseball caps, and winter hats.",
        "Jewelry: Images of decorative accessories, typically made of metal or gemstones. This includes necklaces, earrings, bracelets, rings, brooches, and other forms of fine jewelry.",
        "Watches & Neckwear: Images of watches and accessories worn around the neck. This category includes wristwatches with various bands, as well as neckties and bowties.",
        "Household Goods: Images of items found in a home. This includes a broad range of furniture, kitchenware, home decor, bedding, and appliances like refrigerators and ovens.",
        "Electronics & Gadgets: Images of powered electronic devices and technology. This category includes high-tech items such as smartphones, laptops, televisions, gaming consoles, digital cameras, and smart home devices.",
        "Other: A catch-all category for items that do not belong to the other groups. This includes trash, paper, fabric scraps, packaging, and any background element that is not the main subject of the picture. This category serves as a fallback for any unrelated item."
    ]



    # Every fashion group: apparel, footwear, bags, eyewear,
    # headwear, jewelry, watches & neckwear. 7-9 are rejects.
    direct_accept_indices = [0, 1, 2, 3, 4, 5, 6]

    forbidden_noise_keywords = [
        "paper", "fabric", "scrap", "wrinkled", "crumpled", "trash", "garbage",
        "wrapper", "packaging", "background", "surface", "texture"
    ]

    desc = florence_description.lower()

    try:
        # CLIP inference
        clip_inputs_image = clip_processor(
            text=broad_categories,
            images=image_without_background.convert('RGBA'),
            return_tensors="pt",
            padding=True
        ).to(device)

        with torch.no_grad():
            clip_outputs_image = clip_model(**clip_inputs_image)
            logits_per_image = clip_outputs_image.logits_per_image
            probs = logits_per_image.softmax(dim=1)[0]

            top_idx = probs.argmax().item()
            confidence_score_from_image = probs[top_idx].item()

        # ✅ Direct accept categories
        if top_idx in direct_accept_indices and confidence_score_from_image >= confidence_threshold:
            return True

        if any(word in desc for word in forbidden_noise_keywords):
            return False


        

    except Exception:
        return False


## RUN THE CODE

In [6]:
# original_image_path = "/mnt/c/Users/user/Downloads/New folder/41.png"
original_image_path = "floral pump.png"
# original_image_path = "image.png"

def getting_item_attributes_json(original_image_path):
    extracted_attributes = {}
    processed_image_path = "item_foreground_white_bg.jpg"

    input_image = Image.open(original_image_path).convert("RGB")
    no_bg = remove(input_image)
    del input_image
    item_without_background_path = "item_without_background.png"
    no_bg.save(item_without_background_path)
    bbox = no_bg.getbbox()
    cropped = no_bg.crop(bbox) if bbox else no_bg

    resized = cropped.resize((224, 224), Image.LANCZOS)

    canvas = Image.new("RGB", (224, 224), (255, 255, 255))
    canvas.paste(resized, (0, 0), resized)

    canvas.save("item_for_clip.png")

    item_without_background = Image.open(item_without_background_path).convert("RGB")

    fg = remove_transparency(item_without_background_path)
    fg.save("item_foreground_white_bg.jpg")
    white = Image.open("item_foreground_white_bg.jpg").convert("RGB")

    image_for_description = white

    florence_prompt = "<MORE_DETAILED_CAPTION>"
    florence_inputs = florence_processor(
        text=florence_prompt,
        images=image_for_description,
        return_tensors="pt"
    ).to(device)

    florence_ids = florence_model.generate(
        input_ids=florence_inputs["input_ids"],
        pixel_values=florence_inputs["pixel_values"],
        max_new_tokens=200,
        eos_token_id=florence_processor.tokenizer.eos_token_id,
        do_sample=False
    )

    item_description = florence_processor.batch_decode(
        florence_ids, skip_special_tokens=True
    )[0].strip()

    del florence_inputs, florence_ids
    foreground_image = Image.open(item_without_background_path).convert('RGBA')
    if  not is_item_accepted_with_description(foreground_image, item_description, 0.8):
        torch.cuda.empty_cache()
        gc.collect()
        return ['this is not fashion']
    color_group, color_source, color_confidence = extract_color_group(
        item_description,
        foreground_image,
        n_colors=3
    )

    candidate_labels_category = list(CATEGORY_DESCRIPTIONS.keys())
    clip_inputs = clip_processor(
        text=candidate_labels_category,
        images=foreground_image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        clip_outputs = clip_model(**clip_inputs)
        logits_per_image = clip_outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)[0]

    top_idx = probs.argmax().item()
    predicted_category = candidate_labels_category[top_idx]
    confidence_score_category = probs[top_idx].item()
    del clip_outputs, clip_inputs, probs
    predicted_category, confidence_score_category = correct_category_with_description(
            item_description,
            predicted_category,
            confidence_score_category,
            KEYWORD_TO_CATEGORY,
        )
    predicted_group = CATEGORY_TO_GROUP.get(predicted_category, "Unknown")

    extracted_attributes = {}
    if predicted_group != "Unknown" and predicted_group in ATTRIBUTES:
        group_attributes = ATTRIBUTES[predicted_group]

        for attr_name, attr_candidates in group_attributes.items():
            if not attr_candidates:
                continue
            clip_inputs = clip_processor(
                    text=attr_candidates,
                    images=canvas,
                    return_tensors="pt",
                    padding=True
                ).to(device)

            with torch.no_grad():
                clip_outputs = clip_model(**clip_inputs)
                logits_per_image = clip_outputs.logits_per_image
                probs = logits_per_image.softmax(dim=1)[0]

            top_idx = probs.argmax().item()
            raw_value= attr_candidates[top_idx]
            confidence = probs[top_idx].item()
            del clip_outputs, clip_inputs, probs

            if attr_name == "sleeve":
                concise_value = detect_sleeve_length(
                    description=item_description,
                    attr_candidates=attr_candidates,
                    canvas=canvas,
                    predicted_category=predicted_category
                )

            elif attr_name == "closure":
                if predicted_group == "Footwear":
                    concise_value = detect_footwear_closure(
                        attr_candidates=attr_candidates,
                        canvas=canvas,
                        description=item_description,

                    )
                elif predicted_group == "Tops":
                    concise_value = detect_tops_closure(
                        description=item_description,
                        attr_candidates=attr_candidates,
                        canvas=canvas,

                    )
                elif predicted_group == "Outerwear":

                    concise_value = detect_outerwear_closure(
                        attr_candidates=attr_candidates,
                        canvas=canvas,
                        description=item_description,

                    )

                elif predicted_group == "Bottoms":
                    concise_value = detect_bottoms_closure(
                        canvas=canvas, attr_candidates=attr_candidates,
                        description=item_description,

                    )
                elif predicted_group == "Skirts":
                    concise_value = detect_skirt_closure(
                        attr_candidates=attr_candidates,
                        canvas=canvas,
                        description=item_description,

                    )
                elif predicted_group == "Dresses & Rompers":
                    concise_value = detect_dress_closure(
                        attr_candidates=attr_candidates,
                        canvas=canvas,
                        description=item_description,

                    )
                else :
                    concise_value = detect_accessory_closure(
                        attr_candidates=attr_candidates,
                        canvas=canvas,
                        description=item_description,
                        predicted_group=predicted_group,
                    )


            elif attr_name == "pattern":
                concise_value, confidence  = extract_pattern(
                        item_description,  predicted_group, foreground_image,
                        pattern_keyword_map=pattern_keyword_map
                    )

            elif attr_name == "material":
                concise_value, confidence = extract_material(
                    predicted_group,
                    canvas,
                    description=item_description,
                    predicted_category=predicted_category

                )

            elif attr_name == "length":

                if predicted_group == "Tops":
                    concise_value, method = detect_tops_length(
                        attr_candidates=attr_candidates,
                        image=image_for_description,
                        description=item_description,

                    )


                elif predicted_group == "Outerwear":
                    concise_value, method ,confidence= detect_outerwear_length(
                        attr_candidates=attr_candidates,
                        image=image_for_description,
                        description=item_description,

                    )
                elif predicted_group == "Bottoms":

                    concise_value, method,confidence = detect_bottoms_length(
                        attr_candidates= attr_candidates,
                        image=image_for_description,
                        description=item_description,
                        category=predicted_category,

                    )
                elif predicted_group == "Skirts":
                    concise_value, method,confidence = detect_skirt_length(
                        attr_candidates=attr_candidates,
                        image=image_for_description,
                        description=item_description,

                    )
                elif predicted_group == "Dresses & Rompers":
                    concise_value, method,confidence = detect_dress_length(
                        attr_candidates=attr_candidates,
                        image=image_for_description,
                        description=item_description,
                        category=predicted_category,

                    )

                else:
                    concise_value, method = "unknown", "fallback"

            elif attr_name == "fit":
                if predicted_group == "Tops":
                    concise_value = detect_tops_fit(
                        attr_candidates=attr_candidates,
                        image=image_for_description,
                        description=item_description,

                    )
                elif predicted_group == "Bottoms":
                    concise_value = detect_bottoms_fit(
                        attr_candidates=attr_candidates,
                        image=image_for_description,
                        description=item_description,

                    )
                elif predicted_group == "Skirts":
                    concise_value = detect_skirt_fit(
                        attr_candidates=attr_candidates,
                        image=image_for_description,
                        description=item_description,

                    )
                elif predicted_group == "Underwear & Swimwear":
                    concise_value = detect_underwear_fit(
                        attr_candidates=attr_candidates,
                        image=image_for_description,
                        description=item_description,

                     )
           # elif attr_name == "type":
           #     concise_value = detect_item_type(
           #         predicted_group=predicted_group,
           #         attr_candidates=attr_candidates,
           #         image=image_for_description,
           #         description=item_description,
#
           #       )
            elif attr_name == "coverage":
                concise_value = detect_item_coverage(
                    predicted_group=predicted_group,
                    attr_candidates=attr_candidates,
                    image=image_for_description,
                    description=item_description,

                  )
           # elif attr_name == "style":
           #     concise_value = detect_item_style(
           #         predicted_group=predicted_group,
           #         attr_candidates=attr_candidates,
           #         image=image_for_description,
           #         description=item_description,
#
           #       )

            elif attr_name == "color":
                concise_value = color_group
                source = color_source
                confidence = color_confidence

            else:
                concise_value = raw_value



            if attr_name == "length":
                extracted_attributes[attr_name] = {
                    "value": concise_value,
                    "confidence": round(confidence, 4),
                    "source": method
                }
            elif attr_name == "color":
                extracted_attributes[attr_name] = {
                    "value": concise_value,
                    "confidence": round(confidence, 4),
                    "source": color_source
                }
            else:
                extracted_attributes[attr_name] = {
                    "value": concise_value,
                    "confidence": round(confidence, 4) if confidence is not None else None
                }

        item_map = {}
        item_map['color_group'] = color_group
        item_map['description'] = item_description
        item_map['category'] = predicted_category
        item_map['category_group'] = predicted_group

        for attr_name, data in extracted_attributes.items():

            item_map[attr_name] = data
            if attr_name == "length" and "source" in data:
                conf_str = f"(Source: {data['source']}, Confidence: {data['confidence']:.4f})" if data.get('confidence') is not None else f"(Source: {data['source']})"

            else:
                conf_str = f"(Confidence: {data['confidence']:.4f})" if data.get('confidence') is not None else ""
        cleaned_item_map = {}
        for key, value in item_map.items():
            # إذا dict وفيه 'value'
            if isinstance(value, dict) and 'value' in value:
                v = value['value']
            else:
                v = value

            # إذا tuple أو list → خذ أول عنصر
            if isinstance(v, (list, tuple)):
                v = v[0] if v else None

            # إذا نص فيه " (" → قص
            if isinstance(v, str):
                if '(' in v:
                    v = v.split(' (')[0]
                v = v.strip()

            cleaned_item_map[key] = v
        torch.cuda.empty_cache()
        gc.collect()
        return cleaned_item_map

getting_item_attributes_json(original_image_path)



# for i in range(1, 30):
#     original_image_path = f"/mnt/c/Users/user/Documents/vs code/Fashion4/Python/AI/TEST_IMAGES/{i}.png"

#     getting_item_attributes_json(original_image_path)

{'color_group': 'pastels',
 'description': 'The image shows a pair of women\'s flat shoes. The shoes are light pink in color and have a round toe and a low heel. The upper part of the shoes is covered in a floral print in shades of blue, orange, and yellow. The floral print is made up of small flowers and leaves in various shades of pink, blue, and green. There is a small black label on the side of the shoe with the brand name "JIMMY CHOO" written in white. The shoe appears to be made of a soft, comfortable material and has a slip-on design.',
 'category': 'Ballet flats',
 'category_group': 'Footwear',
 'style': 'flats',
 'closure': 'slip-on',
 'height': 'low-top',
 'toe': 'round toe',
 'pattern': 'floral',
 'material': 'blend'}

# Backend
## If you find the server is not running that's probably because one the group is currently using the NGROK_AUTH_TOKEN
## to fix it
### 1. contact us
## or
### 2. simply go to 'https://dashboard.ngrok.com/get-started/setup/windows' and get your own NGROK_AUTH_TOKEN

In [ ]:
import requests
import os
import tempfile
import uuid
import base64
import time
from pyngrok import ngrok, conf
from flask import Flask, request, jsonify
import os
import tempfile
import base64
import mimetypes
from flask import request, jsonify
from werkzeug.utils import secure_filename

# Credentials come from the environment, never from source.
#   export NGROK_AUTH_TOKEN=...
NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN")
if NGROK_AUTH_TOKEN:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
else:
    print("NGROK_AUTH_TOKEN is not set - running without an ngrok tunnel.")

LARAVEL_URL = os.getenv('LARAVEL_URL', 'http://127.0.0.1:8000')
base_url = LARAVEL_URL

# Default suits Linux/Colab; override on Windows with NGROK_PATH.
ngrok_executable_path = os.getenv('NGROK_PATH', '/usr/local/bin/ngrok2')
conf.get_default().ngrok_path = ngrok_executable_path
print(f"Using ngrok executable from: {ngrok_executable_path}")


app = Flask(__name__)

ALLOWED_EXTENSIONS = {"png", "jpg", "jpeg", "webp", "bmp"}

def _allowed_file(filename: str) -> bool:
    if not filename or "." not in filename:
        return False
    ext = filename.rsplit(".", 1)[1].lower().strip()
    return ext in ALLOWED_EXTENSIONS

def _as_jsonable_attributes(attrs):

    if isinstance(attrs, dict):
        return attrs
    if isinstance(attrs, list):
        if len(attrs) == 1 and isinstance(attrs[0], dict):
            return attrs[0]
        return attrs
    return None

@app.route('/clothingitems', methods=['POST'])
def get_attributes_for_item():
    try:
        print("Received a request to /clothingitems")

        if 'image' not in request.files:
            return jsonify({"error": "We can't do this function without an image. Please upload an image file."}), 400

        image_file = request.files['image']
        if not image_file or image_file.filename == '':
            return jsonify({"error": "Please provide a valid image file with a filename."}), 400

        if not _allowed_file(image_file.filename):
            return jsonify({"error": f"Unsupported file type. Allowed: {', '.join(sorted(ALLOWED_EXTENSIONS))}"}), 400

        safe_name = secure_filename(image_file.filename)

        fd = None
        temp_image_path = None
        try:
            fd, temp_image_path = tempfile.mkstemp(suffix=os.path.splitext(safe_name)[1] or ".png")
            os.close(fd)  
            image_file.save(temp_image_path)
            print(f"Uploaded image saved to temporary path: {temp_image_path}")
        except Exception as ioe:
            
            if fd is not None:
                try:
                    os.close(fd)
                except Exception:
                    pass
            return jsonify({"error": f"Failed to save uploaded image: {ioe}"}), 500

        
        try:
            attributes_raw = getting_item_attributes_json(temp_image_path)
        except Exception as run_err:
            
            print(f"getting_item_attributes_json raised: {run_err}")
            return jsonify({"error": f"Failed to extract attributes: {run_err}"}), 500

        if not attributes_raw:
            return jsonify({"error": "No attributes found for the provided image."}), 400

        attrs = _as_jsonable_attributes(attributes_raw)
        if attrs is None:
            
            return jsonify({"error": "Attributes result has unexpected format."}), 500

        
        processed_image_path = "item_without_background.png"
        image_data_base64 = None
        image_mime = None

        try:
            if os.path.exists(processed_image_path):
                with open(processed_image_path, "rb") as f:
                    image_data_base64 = base64.b64encode(f.read()).decode('utf-8')
                
                image_mime = mimetypes.guess_type(processed_image_path)[0] or "image/png"
                print(f"Processed image read from: {processed_image_path}")
            else:
                
                
                print(f"Processed image file not found at {processed_image_path}. Returning attributes only.")
        except Exception as e_img:
            
            print(f"Error reading processed image: {e_img}")

        response_dict = {
            "data": attrs,
            "imageData": image_data_base64,     
            "imageMimeType": image_mime or "image/png"
        }
        return jsonify(response_dict), 200

    except Exception as e:
        
        print(f"An error occurred during processing: {e}")
        return jsonify({"error": f"An error occurred during processing: {e}"}), 500

    finally:
        try:
            if 'temp_image_path' in locals() and temp_image_path and os.path.exists(temp_image_path):
                os.remove(temp_image_path)
                print(f"Cleaned up temporary file: {temp_image_path}")
        except Exception as cleanup_err:
            print(f"Failed to clean up temporary file: {cleanup_err}")

@app.route('/', methods=['GET'])
def hello_world():
    return "Hello World"

def run_flask_app():
    port = 5001
    try:
        
        app.run(host='0.0.0.0', port=port, debug=False, use_reloader=False)  # Add host='0.0.0.0'
    except Exception as e:
        print(f"Error starting ngrok or Flask app: {e}")

if __name__ == '__main__':
    run_flask_app()

